In [ ]:
# Section 0: Environment check
import torch, torchvision, sys, platform
import numpy as np
import matplotlib.pyplot as plt

print(f"Python:        {sys.version.split()[0]}")
print(f"PyTorch:       {torch.__version__}")
print(f"Torchvision:   {torchvision.__version__}")
print(f"Platform:      {platform.platform()}")

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU:           {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("Running on CPU — training will be very slow.")
print(f"\nDevice selected: {device}")

In [58]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [59]:
# ===== Fix Taiwan dataset structure: merge Train/Test into class folders =====
import shutil

TAIWAN_SRC = "/content/drive/MyDrive/taiwan/Preprocessed data"
TAIWAN_FLAT = "/content/Taiwan_flat"  # local Colab disk for fast access

# Rebuild from scratch each time (idempotent)
if os.path.exists(TAIWAN_FLAT):
    shutil.rmtree(TAIWAN_FLAT)
os.makedirs(TAIWAN_FLAT, exist_ok=True)

# Merge Train/<class>/* and Test/<class>/* into <class>/* in flat dir
for split in ["Train", "Test"]:
    split_path = os.path.join(TAIWAN_SRC, split)
    if not os.path.isdir(split_path):
        continue
    for cls in os.listdir(split_path):
        cls_src = os.path.join(split_path, cls)
        if not os.path.isdir(cls_src):
            continue
        cls_dst = os.path.join(TAIWAN_FLAT, cls)
        os.makedirs(cls_dst, exist_ok=True)
        for fname in os.listdir(cls_src):
            src_file = os.path.join(cls_src, fname)
            dst_file = os.path.join(cls_dst, f"{split}_{fname}")  # prefix to avoid name collision
            shutil.copy(src_file, dst_file)

# Verify
print("Flattened Taiwan dataset structure:")
total = 0
for cls in sorted(os.listdir(TAIWAN_FLAT)):
    n = len(os.listdir(os.path.join(TAIWAN_FLAT, cls)))
    total += n
    print(f"  {cls:<25} {n:>5}")
print(f"  {'TOTAL':<25} {total:>5}")

assert total == 622, f"Expected 622 images, got {total}"
print("\n✓ Structure verified — ready for training.")

# Update TAIWAN_PATH used by the rest of the notebook
TAIWAN_PATH = TAIWAN_FLAT
print(f"\nTAIWAN_PATH updated to: {TAIWAN_PATH}")

Flattened Taiwan dataset structure:
  Bacterial spot              110
  Black mold                   67
  Gray spot                    84
  Late blight                  98
  health                      106
  powdery mildew              157
  TOTAL                       622

✓ Structure verified — ready for training.

TAIWAN_PATH updated to: /content/Taiwan_flat


In [60]:
import os

# ===== Paths =====
# Source dataset (already uploaded to Drive)
# IMPORTANT: adjust this if your folder name differs slightly
TAIWAN_PATH = "/content/drive/MyDrive/taiwan/Preprocessed data"

# Output goes into the SAME project folder as PlantVillage results,
# under a new TAIWAN subdirectory
PROJECT_DIR = "/content/drive/MyDrive/Tomato_3regime"
RESULTS_DIR = f"{PROJECT_DIR}/RESULTS_TAIWAN"

# Reuse the SimCLR checkpoint trained on PlantVillage tomato
SIMCLR_CKPT_SRC = f"{PROJECT_DIR}/RESULTS 2/simclr/simclr_checkpoint.pth"

os.makedirs(f"{RESULTS_DIR}/splits", exist_ok=True)
for method in ["B1_supervised", "B3_self_supervised",
               "B4_semi_supervised", "B5_hybrid"]:
    os.makedirs(f"{RESULTS_DIR}/10pct/{method}", exist_ok=True)

print(f"Source dataset:    {TAIWAN_PATH}")
print(f"Output base:       {RESULTS_DIR}")
print(f"SimCLR checkpoint: {SIMCLR_CKPT_SRC}")

# Sanity check
assert os.path.isdir(TAIWAN_PATH), \
    f"Taiwan dataset not found at {TAIWAN_PATH} — check the path!"
assert os.path.exists(SIMCLR_CKPT_SRC), \
    f"SimCLR checkpoint not found at {SIMCLR_CKPT_SRC} — run PlantVillage Section 8 first."

# Inspect Taiwan classes and image counts
classes_found = sorted([d for d in os.listdir(TAIWAN_PATH)
                        if os.path.isdir(os.path.join(TAIWAN_PATH, d))])
total = 0
print(f"\nFound {len(classes_found)} class folder(s):")
for c in classes_found:
    n = len(os.listdir(os.path.join(TAIWAN_PATH, c)))
    total += n
    print(f"  {c:<35} {n:>5}")
print(f"  {'TOTAL':<35} {total:>5}")

Source dataset:    /content/drive/MyDrive/taiwan/Preprocessed data
Output base:       /content/drive/MyDrive/Tomato_3regime/RESULTS_TAIWAN
SimCLR checkpoint: /content/drive/MyDrive/Tomato_3regime/RESULTS 2/simclr/simclr_checkpoint.pth

Found 2 class folder(s):
  Test                                    6
  Train                                   6
  TOTAL                                  12


In [61]:
# ===== Same hyperparameters as PlantVillage experiments =====
VAL_PCT     = 0.10
IMG_SIZE       = 224
BATCH_SIZE     = 128
NUM_WORKERS    = 2

# SimCLR (not retraining — using existing checkpoint)
SIMCLR_EPOCHS  = 10
LR_SIMCLR      = 1e-4
SIMCLR_TEMP    = 0.5

# MixMatch
K_AUGMENTS      = 2
SHARPEN_TEMP    = 0.5
MIXUP_ALPHA     = 0.75
LAMBDA_U        = 1.0

# Common training duration
EPOCHS_DEFAULT  = 30
SEED            = 42

print(f"Device: {device}  Batch: {BATCH_SIZE}  Img: {IMG_SIZE}")
print(f"Same hyperparameters as PlantVillage runs.")

Device: cuda  Batch: 128  Img: 224
Same hyperparameters as PlantVillage runs.


In [62]:
import random, copy, time, json
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from PIL import Image
from tqdm import tqdm

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def empty_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

base_dataset = ImageFolder(root=TAIWAN_PATH)
NUM_CLASSES  = len(base_dataset.classes)
print(f"Taiwan classes ({NUM_CLASSES}):")
for i, c in enumerate(base_dataset.classes):
    print(f"  {i}: {c}")
print(f"Total images: {len(base_dataset)}")

Taiwan classes (2):
  0: Test
  1: Train
Total images: 622


In [63]:
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(IMG_SIZE, padding=16),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
weak_transform = train_transform

class LabeledSubset(Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base = base_dataset; self.indices = indices; self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        path, label = self.base.samples[self.indices[i]]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label

class UnlabeledKAug(Dataset):
    def __init__(self, base_dataset, indices, transform, K=2):
        self.base = base_dataset; self.indices = indices
        self.transform = transform; self.K = K
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        path, _ = self.base.samples[self.indices[i]]
        img = Image.open(path).convert("RGB")
        return [self.transform(img) for _ in range(self.K)]

def sharpen(p, T=0.5):
    p = p.pow(1.0 / T)
    return p / p.sum(dim=1, keepdim=True)

def mixup(x1, y1, x2, y2, alpha=0.75):
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)
    return lam * x1 + (1 - lam) * x2, lam * y1 + (1 - lam) * y2

print("Transforms and dataset wrappers ready.")

Transforms and dataset wrappers ready.


In [64]:
def stratified_split_with_floor(dataset, label_pct=0.10, val_pct=0.10,
                                 min_per_class=10, seed=42):
    rng = random.Random(seed)
    class_indices = {}
    for idx in range(len(dataset)):
        _, label = dataset.samples[idx]
        class_indices.setdefault(label, []).append(idx)

    labeled_idx, unlabeled_idx, val_idx = [], [], []
    for label, indices in class_indices.items():
        rng.shuffle(indices)
        n = len(indices)
        n_label = max(min_per_class, int(label_pct * n))
        n_val   = max(min_per_class, int(val_pct * n))
        labeled_idx.extend(indices[:n_label])
        val_idx.extend(indices[n_label:n_label + n_val])
        unlabeled_idx.extend(indices[n_label + n_val:])
    return labeled_idx, unlabeled_idx, val_idx

print("Split function defined.")

Split function defined.


In [65]:
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        y_pred.append(logits.argmax(1).cpu().numpy())
        y_true.append(labels.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)

def plot_confusion_matrix(cm, class_names, save_path, title="Confusion Matrix"):
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    for ax, mat, sub, fmt in [(axes[0], cm, "Counts", "d"),
                              (axes[1], cm_norm, "Normalized", ".2f")]:
        im = ax.imshow(mat, cmap="Blues", aspect="auto")
        ax.set_title(f"{title} — {sub}")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names)))
        ax.set_xticklabels([c[:20] for c in class_names], rotation=45, ha="right")
        ax.set_yticklabels([c[:20] for c in class_names])
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = mat[i, j]
                color = "white" if val > mat.max() * 0.55 else "black"
                ax.text(j, i, format(val, fmt), ha="center", va="center",
                        color=color, fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.close(fig)

def plot_training_curves(history, save_path, method_name):
    epochs = history.get("epoch", [])
    has_lx = "loss_x" in history
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    if has_lx:
        axes[0].plot(epochs, history["loss_x"], marker="o", label="labeled (lx)")
        axes[0].plot(epochs, history["loss_u"], marker="s", label="unlabeled (lu)")
    else:
        axes[0].plot(epochs, history["train_loss"], marker="o", label="train loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title(f"{method_name} — training loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(epochs, history["val_acc"], marker="o", color="tab:green")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Val accuracy (%)")
    axes[1].set_title(f"{method_name} — validation accuracy")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.close(fig)

def plot_per_class_bars(per_class, class_names, save_path, method_name):
    n = len(class_names); x = np.arange(n); w = 0.2
    fig, ax = plt.subplots(figsize=(max(10, n * 1.4), 6))
    ax.bar(x - 1.5*w, per_class["accuracy"],  w, label="Accuracy")
    ax.bar(x - 0.5*w, per_class["precision"], w, label="Precision")
    ax.bar(x + 0.5*w, per_class["recall"],    w, label="Recall")
    ax.bar(x + 1.5*w, per_class["f1"],        w, label="F1")
    ax.set_xticks(x)
    ax.set_xticklabels([c[:22] for c in class_names], rotation=30, ha="right")
    ax.set_ylim(0, 1.05); ax.set_ylabel("Score")
    ax.set_title(f"{method_name} — per-class metrics")
    ax.legend(loc="lower right"); ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.close(fig)

def full_evaluation(model, loader, class_names, method_name, history,
                    out_dir, pct_tag, model_state=None):
    os.makedirs(out_dir, exist_ok=True)
    y_true, y_pred = collect_predictions(model, loader, device)
    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    p_weight, r_weight, f1_weight, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0)
    p_cls, r_cls, f1_cls, support = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(class_names)), zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    row_sum = cm.sum(axis=1).clip(min=1)
    acc_cls = cm.diagonal() / row_sum

    per_class = {
        "class_names": list(class_names),
        "accuracy": acc_cls.tolist(), "precision": p_cls.tolist(),
        "recall": r_cls.tolist(), "f1": f1_cls.tolist(),
        "support": support.tolist(),
    }
    metrics = {
        "method": method_name, "pct_tag": pct_tag, "seed": SEED,
        "overall": {
            "accuracy": float(acc),
            "precision_macro": float(p_macro), "recall_macro": float(r_macro),
            "f1_macro": float(f1_macro),
            "precision_weighted": float(p_weight), "recall_weighted": float(r_weight),
            "f1_weighted": float(f1_weight),
        },
        "per_class": per_class, "confusion_matrix": cm.tolist(),
        "history": history,
        "best_val_acc": max(history.get("val_acc", [0.0])),
    }
    with open(f"{out_dir}/metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    report = classification_report(
        y_true, y_pred, labels=range(len(class_names)),
        target_names=class_names, digits=4, zero_division=0)
    with open(f"{out_dir}/classification_report.txt", "w") as f:
        f.write(f"Method: {method_name}   Label %: {pct_tag}   Seed: {SEED}\n")
        f.write("=" * 80 + "\n"); f.write(report)
        f.write("\n\nConfusion matrix (rows=true, cols=pred):\n"); f.write(str(cm))
    plot_confusion_matrix(cm, class_names,
        f"{out_dir}/confusion_matrix.png", title=f"{method_name} @ {pct_tag}")
    plot_training_curves(history,
        f"{out_dir}/training_curves.png", f"{method_name} @ {pct_tag}")
    plot_per_class_bars(per_class, class_names,
        f"{out_dir}/per_class_metrics.png", method_name=f"{method_name} @ {pct_tag}")
    state = model_state if model_state is not None else model.state_dict()
    torch.save({"state_dict": state, "method": method_name,
                "pct_tag": pct_tag, "history": history},
               f"{out_dir}/checkpoint.pth")
    print(f"\n--- {method_name} @ {pct_tag} ---")
    print(f"  Accuracy:           {acc*100:.2f}%")
    print(f"  Precision (macro):  {p_macro*100:.2f}%   (weighted: {p_weight*100:.2f}%)")
    print(f"  Recall    (macro):  {r_macro*100:.2f}%   (weighted: {r_weight*100:.2f}%)")
    print(f"  F1        (macro):  {f1_macro*100:.2f}%   (weighted: {f1_weight*100:.2f}%)")
    print(f"  Saved to: {out_dir}")
    return metrics

print("Metrics utilities loaded.")

Metrics utilities loaded.


In [66]:
class SimCLRModel(nn.Module):
    def __init__(self, feature_dim=128, init="imagenet"):
        super().__init__()
        if init == "imagenet":
            weights = models.ResNet18_Weights.IMAGENET1K_V1
        elif init == "random":
            weights = None
        else:
            raise ValueError(f"Unknown init: {init}")
        self.encoder = models.resnet18(weights=weights)
        self.encoder.fc = nn.Identity()
        self.projector = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Linear(256, feature_dim),
        )
    def forward(self, x):
        h = self.encoder(x); z = self.projector(h)
        return h, z

print("SimCLR model class defined.")

SimCLR model class defined.


In [67]:
simclr_ckpt = torch.load(SIMCLR_CKPT_SRC, map_location=device, weights_only=False)
print(f"Loaded SimCLR checkpoint from PlantVillage pretraining:")
print(f"  Source: {SIMCLR_CKPT_SRC}")
print(f"  Final NT-Xent loss: {simclr_ckpt['final_loss']:.4f}")
print(f"  Epochs: {simclr_ckpt['epoch']}")
print(f"\nNOTE: Using this encoder on Taiwan = cross-dataset transfer (different but related domain).")

Loaded SimCLR checkpoint from PlantVillage pretraining:
  Source: /content/drive/MyDrive/Tomato_3regime/RESULTS 2/simclr/simclr_checkpoint.pth
  Final NT-Xent loss: 3.7202
  Epochs: 10

NOTE: Using this encoder on Taiwan = cross-dataset transfer (different but related domain).


In [68]:
LABEL_PCT = 0.10
pct_tag = f"{int(LABEL_PCT*100)}pct"
CUR_OUT = f"{RESULTS_DIR}/{pct_tag}"
print(f"Active experiment: LABEL_PCT={LABEL_PCT}  (tag: {pct_tag})")
print(f"Output base:       {CUR_OUT}")

set_seed(SEED)
labeled_idx, unlabeled_idx, val_idx = stratified_split_with_floor(
    base_dataset, label_pct=LABEL_PCT, val_pct=VAL_PCT,
    min_per_class=10, seed=SEED,
)

total_n = len(base_dataset)
print(f"\nSplit sizes (target label_pct={LABEL_PCT}, val_pct={VAL_PCT}, floor=10):")
print(f"  Labeled:    {len(labeled_idx):>5} ({100*len(labeled_idx)/total_n:.2f}%)")
print(f"  Unlabeled:  {len(unlabeled_idx):>5} ({100*len(unlabeled_idx)/total_n:.2f}%)")
print(f"  Validation: {len(val_idx):>5} ({100*len(val_idx)/total_n:.2f}%)")

with open(f"{RESULTS_DIR}/splits/split_{pct_tag}_seed{SEED}.json", "w") as f:
    json.dump({
        "seed": SEED, "label_pct": LABEL_PCT, "val_pct": VAL_PCT,
        "min_per_class": 10,
        "labeled_idx": labeled_idx, "unlabeled_idx": unlabeled_idx,
        "val_idx": val_idx, "classes": base_dataset.classes,
    }, f)

labeled_dataset   = LabeledSubset(base_dataset, labeled_idx,   weak_transform)
unlabeled_dataset = UnlabeledKAug(base_dataset, unlabeled_idx, weak_transform, K=K_AUGMENTS)
val_dataset       = LabeledSubset(base_dataset, val_idx,       eval_transform)

# CRITICAL: drop_last=False on labeled_loader because labeled set < batch size
# Without this, MixMatch/B5 would loop over zero batches
labeled_loader = DataLoader(labeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
train_loader = DataLoader(labeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)

print(f"\nLoader batch counts:")
print(f"  labeled_loader:   {len(labeled_loader)}   (drop_last=False for small dataset)")
print(f"  unlabeled_loader: {len(unlabeled_loader)}")
print(f"  val_loader:       {len(val_loader)}")
print(f"  train_loader:     {len(train_loader)}")

results_cur = {}

Active experiment: LABEL_PCT=0.1  (tag: 10pct)
Output base:       /content/drive/MyDrive/Tomato_3regime/RESULTS_TAIWAN/10pct

Split sizes (target label_pct=0.1, val_pct=0.1, floor=10):
  Labeled:       61 (9.81%)
  Unlabeled:    500 (80.39%)
  Validation:    61 (9.81%)

Loader batch counts:
  labeled_loader:   1   (drop_last=False for small dataset)
  unlabeled_loader: 3
  val_loader:       1
  train_loader:     1


In [69]:
set_seed(SEED)
sup_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
sup_model.fc = nn.Linear(512, NUM_CLASSES)
sup_model = sup_model.to(device)

LR_SUP = 1e-3
optimizer = torch.optim.Adam(sup_model.parameters(), lr=LR_SUP)
criterion = nn.CrossEntropyLoss()
EPOCHS = EPOCHS_DEFAULT

print("=" * 64); print(f"B1 @ {pct_tag} (Taiwan): Supervised (ImageNet init)"); print("=" * 64)

b1_history = {"epoch": [], "train_loss": [], "val_acc": []}
best, best_state = 0.0, None
start = time.time()

for epoch in range(EPOCHS):
    sup_model.train()
    tl, nb = 0.0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        loss = criterion(sup_model(imgs), labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tl += loss.item(); nb += 1
    sup_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (sup_model(imgs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = 100 * correct / total
    b1_history["epoch"].append(epoch + 1)
    b1_history["train_loss"].append(tl / nb)
    b1_history["val_acc"].append(val_acc)
    if val_acc > best:
        best = val_acc; best_state = copy.deepcopy(sup_model.state_dict())
    print(f"  Ep {epoch+1:>2}/{EPOCHS}  loss={tl/nb:.4f}  val={val_acc:.2f}%  (best: {best:.2f}%)")

print(f"\nB1 done. Time: {(time.time()-start)/60:.1f}m | Best: {best:.2f}%")
sup_model.load_state_dict(best_state)
results_cur["B1_supervised"] = best
metrics_b1 = full_evaluation(
    model=sup_model, loader=val_loader, class_names=base_dataset.classes,
    method_name="B1_supervised", history=b1_history,
    out_dir=f"{CUR_OUT}/B1_supervised", pct_tag=pct_tag,
)
del sup_model, optimizer; empty_cache()

B1 @ 10pct (Taiwan): Supervised (ImageNet init)
  Ep  1/30  loss=0.6256  val=63.93%  (best: 63.93%)
  Ep  2/30  loss=0.2202  val=31.15%  (best: 63.93%)
  Ep  3/30  loss=0.1358  val=62.30%  (best: 63.93%)


KeyboardInterrupt: 

In [ ]:
class SimCLRFineTune(nn.Module):
    def __init__(self, pretrained_encoder, num_classes):
        super().__init__()
        self.encoder = copy.deepcopy(pretrained_encoder)
        self.classifier = nn.Linear(512, num_classes)
    def forward(self, x):
        return self.classifier(self.encoder(x))

set_seed(SEED)
simclr_clean = SimCLRModel(feature_dim=128, init="imagenet").to(device)
simclr_clean.load_state_dict(simclr_ckpt["model_state_dict"])

ft_model = SimCLRFineTune(simclr_clean.encoder, NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(ft_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
EPOCHS = EPOCHS_DEFAULT

print("=" * 64); print(f"B3 @ {pct_tag} (Taiwan): SimCLR + Full FT (LR=1e-4)"); print("=" * 64)

b3_history = {"epoch": [], "train_loss": [], "val_acc": []}
best, best_state = 0.0, None
start = time.time()

for epoch in range(EPOCHS):
    ft_model.train()
    tl, nb = 0.0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        loss = criterion(ft_model(imgs), labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tl += loss.item(); nb += 1
    ft_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (ft_model(imgs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = 100 * correct / total
    b3_history["epoch"].append(epoch + 1)
    b3_history["train_loss"].append(tl / nb)
    b3_history["val_acc"].append(val_acc)
    if val_acc > best:
        best = val_acc; best_state = copy.deepcopy(ft_model.state_dict())
    print(f"  Ep {epoch+1:>2}/{EPOCHS}  loss={tl/nb:.3f}  val={val_acc:.2f}%  (best: {best:.2f}%)")

print(f"\nB3 done. Time: {(time.time()-start)/60:.1f}m | Best: {best:.2f}%")
ft_model.load_state_dict(best_state)
results_cur["B3_self_supervised"] = best
metrics_b3 = full_evaluation(
    model=ft_model, loader=val_loader, class_names=base_dataset.classes,
    method_name="B3_self_supervised", history=b3_history,
    out_dir=f"{CUR_OUT}/B3_self_supervised", pct_tag=pct_tag,
)
del ft_model, optimizer, simclr_clean; empty_cache()

In [ ]:
set_seed(SEED)
mm_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
mm_model.fc = nn.Linear(512, NUM_CLASSES)
mm_model = mm_model.to(device)

LR_MM = 1e-4
optimizer = torch.optim.Adam(mm_model.parameters(), lr=LR_MM)
EPOCHS = EPOCHS_DEFAULT

print("=" * 64); print(f"B4 @ {pct_tag} (Taiwan): MixMatch (ImageNet init)"); print("=" * 64)

b4_history = {"epoch": [], "loss_x": [], "loss_u": [], "val_acc": []}
best, best_state = 0.0, None
start = time.time()

for epoch in range(EPOCHS):
    mm_model.train()
    tlx, tlu, nb = 0.0, 0.0, 0
    uiter = iter(unlabeled_loader)
    for bx, by in labeled_loader:
        try: buv = next(uiter)
        except StopIteration:
            uiter = iter(unlabeled_loader); buv = next(uiter)
        bx, by = bx.to(device), by.to(device)
        buv = [v.to(device) for v in buv]
        with torch.no_grad():
            probs = torch.zeros(buv[0].size(0), NUM_CLASSES, device=device)
            for v in buv: probs += F.softmax(mm_model(v), dim=1)
            probs /= len(buv)
            pseudo = sharpen(probs, T=SHARPEN_TEMP)
        loh = F.one_hot(by, num_classes=NUM_CLASSES).float()
        ai = torch.cat([bx] + buv, dim=0)
        at = torch.cat([loh] + [pseudo] * len(buv), dim=0)
        si = torch.randperm(ai.size(0))
        mx, my = mixup(ai, at, ai[si], at[si], alpha=MIXUP_ALPHA)
        n = bx.size(0)
        xp, yp = mx[:n], my[:n]; up, uy = mx[n:], my[n:]
        lx = -torch.mean(torch.sum(yp * F.log_softmax(mm_model(xp), dim=1), dim=1))
        lu = F.mse_loss(F.softmax(mm_model(up), dim=1), uy)
        loss = lx + LAMBDA_U * lu
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tlx += lx.item(); tlu += lu.item(); nb += 1
    mm_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (mm_model(imgs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = 100 * correct / total
    b4_history["epoch"].append(epoch + 1)
    b4_history["loss_x"].append(tlx / nb); b4_history["loss_u"].append(tlu / nb)
    b4_history["val_acc"].append(val_acc)
    if val_acc > best:
        best = val_acc; best_state = copy.deepcopy(mm_model.state_dict())
    print(f"  Ep {epoch+1:>2}/{EPOCHS}  lx={tlx/nb:.3f}  lu={tlu/nb:.4f}  "
          f"val={val_acc:.2f}%  (best: {best:.2f}%)")

print(f"\nB4 done. Time: {(time.time()-start)/60:.1f}m | Best: {best:.2f}%")
mm_model.load_state_dict(best_state)
results_cur["B4_semi_supervised"] = best
metrics_b4 = full_evaluation(
    model=mm_model, loader=val_loader, class_names=base_dataset.classes,
    method_name="B4_semi_supervised", history=b4_history,
    out_dir=f"{CUR_OUT}/B4_semi_supervised", pct_tag=pct_tag,
)
del mm_model, optimizer; empty_cache()

In [ ]:
class SimCLRMixMatchFull(nn.Module):
    def __init__(self, pretrained_encoder, num_classes):
        super().__init__()
        self.encoder = copy.deepcopy(pretrained_encoder)
        self.classifier = nn.Linear(512, num_classes)
    def forward(self, x):
        return self.classifier(self.encoder(x))

set_seed(SEED)
simclr_clean = SimCLRModel(feature_dim=128, init="imagenet").to(device)
simclr_clean.load_state_dict(simclr_ckpt["model_state_dict"])

b5_full = SimCLRMixMatchFull(simclr_clean.encoder, NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(b5_full.parameters(), lr=1e-4)
EPOCHS = EPOCHS_DEFAULT

print("=" * 64); print(f"B5-full @ {pct_tag} (Taiwan): SimCLR + MixMatch full FT"); print("=" * 64)

b5_history = {"epoch": [], "loss_x": [], "loss_u": [], "val_acc": []}
best, best_state = 0.0, None
start = time.time()

for epoch in range(EPOCHS):
    b5_full.train()
    tlx, tlu, nb = 0.0, 0.0, 0
    uiter = iter(unlabeled_loader)
    for bx, by in labeled_loader:
        try: buv = next(uiter)
        except StopIteration:
            uiter = iter(unlabeled_loader); buv = next(uiter)
        bx, by = bx.to(device), by.to(device)
        buv = [v.to(device) for v in buv]
        with torch.no_grad():
            probs = torch.zeros(buv[0].size(0), NUM_CLASSES, device=device)
            for v in buv: probs += F.softmax(b5_full(v), dim=1)
            probs /= len(buv)
            pseudo = sharpen(probs, T=SHARPEN_TEMP)
        loh = F.one_hot(by, num_classes=NUM_CLASSES).float()
        ai = torch.cat([bx] + buv, dim=0)
        at = torch.cat([loh] + [pseudo] * len(buv), dim=0)
        si = torch.randperm(ai.size(0))
        mx, my = mixup(ai, at, ai[si], at[si], alpha=MIXUP_ALPHA)
        n = bx.size(0)
        xp, yp = mx[:n], my[:n]; up, uy = mx[n:], my[n:]
        lx = -torch.mean(torch.sum(yp * F.log_softmax(b5_full(xp), dim=1), dim=1))
        lu = F.mse_loss(F.softmax(b5_full(up), dim=1), uy)
        loss = lx + LAMBDA_U * lu
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tlx += lx.item(); tlu += lu.item(); nb += 1
    b5_full.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (b5_full(imgs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = 100 * correct / total
    b5_history["epoch"].append(epoch + 1)
    b5_history["loss_x"].append(tlx / nb); b5_history["loss_u"].append(tlu / nb)
    b5_history["val_acc"].append(val_acc)
    if val_acc > best:
        best = val_acc; best_state = copy.deepcopy(b5_full.state_dict())
    print(f"  Ep {epoch+1:>2}/{EPOCHS}  lx={tlx/nb:.3f}  lu={tlu/nb:.4f}  "
          f"val={val_acc:.2f}%  (best: {best:.2f}%)")

print(f"\nB5-full done. Time: {(time.time()-start)/60:.1f}m | Best: {best:.2f}%")
b5_full.load_state_dict(best_state)
results_cur["B5_hybrid"] = best
metrics_b5 = full_evaluation(
    model=b5_full, loader=val_loader, class_names=base_dataset.classes,
    method_name="B5_hybrid", history=b5_history,
    out_dir=f"{CUR_OUT}/B5_hybrid", pct_tag=pct_tag,
)
del b5_full, optimizer, simclr_clean; empty_cache()

print("\n" + "=" * 64); print(f"SUMMARY @ {pct_tag} (Taiwan)"); print("=" * 64)
for method, acc in results_cur.items():
    print(f"  {method:<25}  {acc:.2f}%")

In [ ]:
import csv

METHOD_ORDER = ["B1_supervised", "B3_self_supervised", "B4_semi_supervised", "B5_hybrid"]
METHOD_DISPLAY = {
    "B1_supervised":      "B1: Supervised",
    "B3_self_supervised": "B3: SimCLR + FT",
    "B4_semi_supervised": "B4: MixMatch",
    "B5_hybrid":          "B5: SimCLR + MixMatch",
}

all_metrics = {}
for method in METHOD_ORDER:
    path = f"{CUR_OUT}/{method}/metrics.json"
    if os.path.exists(path):
        with open(path) as f:
            all_metrics[method] = json.load(f)

class_names = base_dataset.classes
n_classes  = len(class_names)
n_methods  = len(METHOD_ORDER)
save_dir   = f"{CUR_OUT}/_comparison"
os.makedirs(save_dir, exist_ok=True)

# Overall metrics figure
metric_keys = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
metric_labels = ["Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)"]
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(n_methods); w = 0.2
for j, (mk, ml) in enumerate(zip(metric_keys, metric_labels)):
    vals = [all_metrics[m]["overall"][mk] for m in METHOD_ORDER]
    bars = ax.bar(x + (j - 1.5) * w, vals, w, label=ml)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + 0.005,
                f"{v*100:.1f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels([METHOD_DISPLAY[m] for m in METHOD_ORDER], rotation=15, ha="right")
ax.set_ylim(0, 1.08); ax.set_ylabel("Score")
ax.set_title(f"Overall metrics — Taiwan dataset @ {pct_tag}")
ax.legend(loc="lower right"); ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{save_dir}/overall_metrics_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# Per-class F1 heatmap
f1_matrix = np.zeros((n_methods, n_classes))
for i, m in enumerate(METHOD_ORDER):
    f1_matrix[i] = all_metrics[m]["per_class"]["f1"]
fig, ax = plt.subplots(figsize=(max(10, n_classes * 1.4), 5))
im = ax.imshow(f1_matrix, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(n_classes))
ax.set_xticklabels([c[:22] for c in class_names], rotation=30, ha="right")
ax.set_yticks(range(n_methods))
ax.set_yticklabels([METHOD_DISPLAY[m] for m in METHOD_ORDER])
for i in range(n_methods):
    for j in range(n_classes):
        v = f1_matrix[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                color="white" if v < 0.5 else "black", fontsize=9)
ax.set_title(f"Per-class F1 — Taiwan @ {pct_tag}")
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01, label="F1 score")
plt.tight_layout()
plt.savefig(f"{save_dir}/per_class_f1_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# Print + save comparison table including TOM-SSL benchmark
print("\n" + "=" * 80); print(f"TAIWAN COMPARISON @ {pct_tag}"); print("=" * 80)
print(f"  {'Method':<28} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}")
print("  " + "-" * 62)
rows = []
for m in METHOD_ORDER:
    o = all_metrics[m]["overall"]
    rows.append([METHOD_DISPLAY[m], o["accuracy"]*100, o["precision_macro"]*100,
                 o["recall_macro"]*100, o["f1_macro"]*100])
    print(f"  {METHOD_DISPLAY[m]:<28} {o['accuracy']*100:>7.2f}% "
          f"{o['precision_macro']*100:>7.2f}% {o['recall_macro']*100:>7.2f}% "
          f"{o['f1_macro']*100:>7.2f}%")

# Add TOM-SSL paper benchmarks (from their Table 2)
print("\n  --- TOM-SSL paper benchmarks (augmented Taiwan, 10% labels) ---")
print(f"  {'TOM-SSL (their paper)':<28} {'70.87':>7}% {'—':>7}  {'—':>7}  {'69.28':>7}%")
print(f"  {'MixMatch (their paper)':<28} {'67.26':>7}% {'—':>7}  {'—':>7}  {'65.43':>7}%")
print(f"  {'MobileNetV3-Sup (theirs)':<28} {'56.52':>7}% {'—':>7}  {'—':>7}  {'53.73':>7}%")
print("\n  NOTE: Their numbers are on the AUGMENTED Taiwan dataset (4,976 imgs).")
print("        Ours are on PREPROCESSED original (~622 imgs). Direct comparison")
print("        is NOT apples-to-apples — different effective label count and dataset.")

with open(f"{save_dir}/comparison_table.csv", "w", newline="") as f:
    w_csv = csv.writer(f)
    w_csv.writerow(["Method","Accuracy","Precision_macro","Recall_macro","F1_macro","Source"])
    for r in rows:
        w_csv.writerow([r[0]] + [f"{x:.4f}" for x in r[1:]] + ["ours_preprocessed_622"])
    w_csv.writerow(["TOM-SSL", "0.7087", "—", "—", "0.6928", "tomssl_paper_augmented_4976"])
    w_csv.writerow(["MixMatch_tomssl", "0.6726", "—", "—", "0.6543", "tomssl_paper_augmented_4976"])
    w_csv.writerow(["MobileNetV3_sup", "0.5652", "—", "—", "0.5373", "tomssl_paper_augmented_4976"])

print(f"\nComparison saved to: {save_dir}")

In [ ]:
import os
AUG_BASE = "/content/drive/MyDrive/taiwan/data augmentation"

def show_tree(path, depth=0, max_depth=3):
    if depth > max_depth: return
    indent = "  " * depth
    name = os.path.basename(path) or path
    if os.path.isdir(path):
        entries = sorted(os.listdir(path))
        non_dirs = [e for e in entries if not os.path.isdir(os.path.join(path, e))]
        subdirs  = [e for e in entries if os.path.isdir(os.path.join(path, e))]
        if len(non_dirs) > 5 and len(subdirs) == 0:
            print(f"{indent}{name}/  [{len(non_dirs)} files]")
        else:
            print(f"{indent}{name}/")
            for sub in subdirs:
                show_tree(os.path.join(path, sub), depth + 1, max_depth)

show_tree(AUG_BASE)

In [ ]:
# ===== Flatten augmented Taiwan dataset (Train + Test → single class folders) =====
AUG_SRC  = "/content/drive/MyDrive/taiwan/data augmentation"
AUG_FLAT = "/content/Taiwan_aug_flat"

if os.path.exists(AUG_FLAT):
    shutil.rmtree(AUG_FLAT)
os.makedirs(AUG_FLAT, exist_ok=True)

for split in ["Train", "Test"]:
    split_path = os.path.join(AUG_SRC, split)
    if not os.path.isdir(split_path):
        continue
    for cls in os.listdir(split_path):
        cls_src = os.path.join(split_path, cls)
        if not os.path.isdir(cls_src):
            continue
        cls_dst = os.path.join(AUG_FLAT, cls)
        os.makedirs(cls_dst, exist_ok=True)
        for fname in os.listdir(cls_src):
            src_file = os.path.join(cls_src, fname)
            dst_file = os.path.join(cls_dst, f"{split}_{fname}")
            shutil.copy(src_file, dst_file)

print("Flattened augmented Taiwan dataset:")
total = 0
for cls in sorted(os.listdir(AUG_FLAT)):
    n = len(os.listdir(os.path.join(AUG_FLAT, cls)))
    total += n
    print(f"  {cls:<25} {n:>5}")
print(f"  {'TOTAL':<25} {total:>5}")
assert total == 4976
print(f"\n✓ Augmented Taiwan dataset ready at: {AUG_FLAT}")

In [ ]:
# ===== Inspect filenames to derive a source-image grouping rule =====
sample_class = sorted(os.listdir(AUG_FLAT))[0]  # any class will do
sample_files = sorted(os.listdir(os.path.join(AUG_FLAT, sample_class)))[:20]

print(f"First 20 filenames in class '{sample_class}':")
for f in sample_files:
    print(f"  {f}")

In [ ]:
# Verify grouping pattern across all 6 classes
import re

def source_id_from_filename(fname):
    """
    Extract the source-image identifier.
    Filenames look like 'Test_Bs105_mirror.jpg' or 'Train_Bs105.JPG'.
    Source ID = everything up to (and including) the digits after the
    class-letter prefix.
    """
    # Strip extension
    stem = os.path.splitext(fname)[0]
    # Pattern: <Split>_<Letters><Digits>(_optional suffix)
    m = re.match(r"^(Train|Test)_([A-Za-z]+\d+)", stem)
    if m:
        return f"{m.group(1)}_{m.group(2)}"
    # Fallback: treat whole stem as its own group (no augmented siblings)
    return stem

# Check each class's first few files
for cls in sorted(os.listdir(AUG_FLAT)):
    files = sorted(os.listdir(os.path.join(AUG_FLAT, cls)))
    print(f"\n=== {cls} ({len(files)} files) ===")
    # Show 8 filenames and their derived source IDs
    for f in files[:8]:
        print(f"  {f:<45} → source_id: {source_id_from_filename(f)}")
    # Count unique source images in this class
    unique_sources = set(source_id_from_filename(f) for f in files)
    print(f"  Unique source images: {len(unique_sources)}  (avg {len(files)/len(unique_sources):.1f} copies each)")

# Grand totals
all_sources = set()
for cls in sorted(os.listdir(AUG_FLAT)):
    for f in os.listdir(os.path.join(AUG_FLAT, cls)):
        all_sources.add(source_id_from_filename(f))
print(f"\n=== TOTAL ===")
print(f"  Total files:           4976")
print(f"  Unique source images:  {len(all_sources)}  (expected ~622)")

In [ ]:
# ===== Experiment A: Augmented Taiwan + naive random split (TOM-SSL protocol) =====
# This mirrors TOM-SSL's setup: 10% stratified random sample from the augmented pool.
# Augmented copies of the same source image will land in both train and val (leakage).

RESULTS_AUG_NAIVE = f"{PROJECT_DIR}/RESULTS_TAIWAN_AUG_NAIVE"
for method in ["B1_supervised", "B3_self_supervised",
               "B4_semi_supervised", "B5_hybrid"]:
    os.makedirs(f"{RESULTS_AUG_NAIVE}/10pct/{method}", exist_ok=True)
os.makedirs(f"{RESULTS_AUG_NAIVE}/splits", exist_ok=True)

# Switch base_dataset to augmented flat
base_dataset = ImageFolder(root=AUG_FLAT)
NUM_CLASSES  = len(base_dataset.classes)
print(f"Switched to augmented Taiwan dataset:")
print(f"  Classes ({NUM_CLASSES}): {base_dataset.classes}")
print(f"  Total images: {len(base_dataset)}")

In [ ]:
# ===== Build naive (random) 10% split — TOM-SSL style =====
LABEL_PCT = 0.10
pct_tag = f"{int(LABEL_PCT*100)}pct"
CUR_OUT = f"{RESULTS_AUG_NAIVE}/{pct_tag}"
print(f"Experiment A (naive split): LABEL_PCT={LABEL_PCT}")
print(f"Output base: {CUR_OUT}")

set_seed(SEED)
labeled_idx, unlabeled_idx, val_idx = stratified_split_with_floor(
    base_dataset, label_pct=LABEL_PCT, val_pct=VAL_PCT,
    min_per_class=10, seed=SEED,
)

total_n = len(base_dataset)
print(f"\nSplit sizes:")
print(f"  Labeled:    {len(labeled_idx):>5} ({100*len(labeled_idx)/total_n:.2f}%)")
print(f"  Unlabeled:  {len(unlabeled_idx):>5} ({100*len(unlabeled_idx)/total_n:.2f}%)")
print(f"  Validation: {len(val_idx):>5} ({100*len(val_idx)/total_n:.2f}%)")

# ===== Leakage diagnostic — how many val source-IDs also appear in labeled? =====
def src_of(idx):
    path, _ = base_dataset.samples[idx]
    return source_id_from_filename(os.path.basename(path))

labeled_sources = set(src_of(i) for i in labeled_idx)
val_sources     = set(src_of(i) for i in val_idx)
overlap = labeled_sources & val_sources
print(f"\n=== LEAKAGE DIAGNOSTIC (naive split) ===")
print(f"  Distinct source IDs in labeled set:    {len(labeled_sources)}")
print(f"  Distinct source IDs in validation set: {len(val_sources)}")
print(f"  Source IDs appearing in BOTH:          {len(overlap)} "
      f"({100*len(overlap)/len(val_sources):.1f}% of val sources)")
print(f"  → This is the leakage. {len(overlap)} val images have a sibling in train.")

with open(f"{RESULTS_AUG_NAIVE}/splits/split_{pct_tag}_seed{SEED}.json", "w") as f:
    json.dump({
        "seed": SEED, "label_pct": LABEL_PCT, "protocol": "naive_random",
        "labeled_idx": labeled_idx, "unlabeled_idx": unlabeled_idx,
        "val_idx": val_idx,
        "leakage_overlap_count": len(overlap),
        "classes": base_dataset.classes,
    }, f)

# Build loaders (drop_last=False because labeled set may be small per batch)
labeled_dataset   = LabeledSubset(base_dataset, labeled_idx,   weak_transform)
unlabeled_dataset = UnlabeledKAug(base_dataset, unlabeled_idx, weak_transform, K=K_AUGMENTS)
val_dataset       = LabeledSubset(base_dataset, val_idx,       eval_transform)

labeled_loader = DataLoader(labeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
train_loader = DataLoader(labeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)

print(f"\nLoader batch counts:")
print(f"  labeled:   {len(labeled_loader)}")
print(f"  unlabeled: {len(unlabeled_loader)}")
print(f"  val:       {len(val_loader)}")
print(f"  train:     {len(train_loader)}")

results_cur = {}

In [ ]:
# ===== B1: Supervised (ImageNet init) =====
set_seed(SEED)
sup_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
sup_model.fc = nn.Linear(512, NUM_CLASSES)
sup_model = sup_model.to(device)

LR_SUP = 1e-3
optimizer = torch.optim.Adam(sup_model.parameters(), lr=LR_SUP)
criterion = nn.CrossEntropyLoss()
EPOCHS = EPOCHS_DEFAULT

print("=" * 64); print(f"B1 @ {pct_tag} (Aug-Naive): Supervised (ImageNet init)"); print("=" * 64)

b1_history = {"epoch": [], "train_loss": [], "val_acc": []}
best, best_state = 0.0, None
start = time.time()

for epoch in range(EPOCHS):
    sup_model.train()
    tl, nb = 0.0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        loss = criterion(sup_model(imgs), labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tl += loss.item(); nb += 1
    sup_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (sup_model(imgs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = 100 * correct / total
    b1_history["epoch"].append(epoch + 1)
    b1_history["train_loss"].append(tl / nb)
    b1_history["val_acc"].append(val_acc)
    if val_acc > best:
        best = val_acc; best_state = copy.deepcopy(sup_model.state_dict())
    print(f"  Ep {epoch+1:>2}/{EPOCHS}  loss={tl/nb:.4f}  val={val_acc:.2f}%  (best: {best:.2f}%)")

print(f"\nB1 done. Time: {(time.time()-start)/60:.1f}m | Best: {best:.2f}%")
sup_model.load_state_dict(best_state)
results_cur["B1_supervised"] = best
metrics_b1 = full_evaluation(
    model=sup_model, loader=val_loader, class_names=base_dataset.classes,
    method_name="B1_supervised", history=b1_history,
    out_dir=f"{CUR_OUT}/B1_supervised", pct_tag=pct_tag,
)
del sup_model, optimizer; empty_cache()

In [ ]:
# ===== B3: SimCLR + Full Fine-tune =====
class SimCLRFineTune(nn.Module):
    def __init__(self, pretrained_encoder, num_classes):
        super().__init__()
        self.encoder = copy.deepcopy(pretrained_encoder)
        self.classifier = nn.Linear(512, num_classes)
    def forward(self, x):
        return self.classifier(self.encoder(x))

set_seed(SEED)
simclr_clean = SimCLRModel(feature_dim=128, init="imagenet").to(device)
simclr_clean.load_state_dict(simclr_ckpt["model_state_dict"])

ft_model = SimCLRFineTune(simclr_clean.encoder, NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(ft_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
EPOCHS = EPOCHS_DEFAULT

print("=" * 64); print(f"B3 @ {pct_tag} (Aug-Naive): SimCLR + Full FT (LR=1e-4)"); print("=" * 64)

b3_history = {"epoch": [], "train_loss": [], "val_acc": []}
best, best_state = 0.0, None
start = time.time()

for epoch in range(EPOCHS):
    ft_model.train()
    tl, nb = 0.0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        loss = criterion(ft_model(imgs), labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tl += loss.item(); nb += 1
    ft_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (ft_model(imgs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = 100 * correct / total
    b3_history["epoch"].append(epoch + 1)
    b3_history["train_loss"].append(tl / nb)
    b3_history["val_acc"].append(val_acc)
    if val_acc > best:
        best = val_acc; best_state = copy.deepcopy(ft_model.state_dict())
    print(f"  Ep {epoch+1:>2}/{EPOCHS}  loss={tl/nb:.3f}  val={val_acc:.2f}%  (best: {best:.2f}%)")

print(f"\nB3 done. Time: {(time.time()-start)/60:.1f}m | Best: {best:.2f}%")
ft_model.load_state_dict(best_state)
results_cur["B3_self_supervised"] = best
metrics_b3 = full_evaluation(
    model=ft_model, loader=val_loader, class_names=base_dataset.classes,
    method_name="B3_self_supervised", history=b3_history,
    out_dir=f"{CUR_OUT}/B3_self_supervised", pct_tag=pct_tag,
)
del ft_model, optimizer, simclr_clean; empty_cache()

In [ ]:
# ===== B4: MixMatch (ImageNet init) =====
set_seed(SEED)
mm_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
mm_model.fc = nn.Linear(512, NUM_CLASSES)
mm_model = mm_model.to(device)

LR_MM = 1e-4
optimizer = torch.optim.Adam(mm_model.parameters(), lr=LR_MM)
EPOCHS = EPOCHS_DEFAULT

print("=" * 64); print(f"B4 @ {pct_tag} (Aug-Naive): MixMatch (ImageNet init)"); print("=" * 64)

b4_history = {"epoch": [], "loss_x": [], "loss_u": [], "val_acc": []}
best, best_state = 0.0, None
start = time.time()

for epoch in range(EPOCHS):
    mm_model.train()
    tlx, tlu, nb = 0.0, 0.0, 0
    uiter = iter(unlabeled_loader)
    for bx, by in labeled_loader:
        try: buv = next(uiter)
        except StopIteration:
            uiter = iter(unlabeled_loader); buv = next(uiter)
        bx, by = bx.to(device), by.to(device)
        buv = [v.to(device) for v in buv]
        with torch.no_grad():
            probs = torch.zeros(buv[0].size(0), NUM_CLASSES, device=device)
            for v in buv: probs += F.softmax(mm_model(v), dim=1)
            probs /= len(buv)
            pseudo = sharpen(probs, T=SHARPEN_TEMP)
        loh = F.one_hot(by, num_classes=NUM_CLASSES).float()
        ai = torch.cat([bx] + buv, dim=0)
        at = torch.cat([loh] + [pseudo] * len(buv), dim=0)
        si = torch.randperm(ai.size(0))
        mx, my = mixup(ai, at, ai[si], at[si], alpha=MIXUP_ALPHA)
        n = bx.size(0)
        xp, yp = mx[:n], my[:n]; up, uy = mx[n:], my[n:]
        lx = -torch.mean(torch.sum(yp * F.log_softmax(mm_model(xp), dim=1), dim=1))
        lu = F.mse_loss(F.softmax(mm_model(up), dim=1), uy)
        loss = lx + LAMBDA_U * lu
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tlx += lx.item(); tlu += lu.item(); nb += 1
    mm_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (mm_model(imgs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = 100 * correct / total
    b4_history["epoch"].append(epoch + 1)
    b4_history["loss_x"].append(tlx / nb); b4_history["loss_u"].append(tlu / nb)
    b4_history["val_acc"].append(val_acc)
    if val_acc > best:
        best = val_acc; best_state = copy.deepcopy(mm_model.state_dict())
    print(f"  Ep {epoch+1:>2}/{EPOCHS}  lx={tlx/nb:.3f}  lu={tlu/nb:.4f}  "
          f"val={val_acc:.2f}%  (best: {best:.2f}%)")

print(f"\nB4 done. Time: {(time.time()-start)/60:.1f}m | Best: {best:.2f}%")
mm_model.load_state_dict(best_state)
results_cur["B4_semi_supervised"] = best
metrics_b4 = full_evaluation(
    model=mm_model, loader=val_loader, class_names=base_dataset.classes,
    method_name="B4_semi_supervised", history=b4_history,
    out_dir=f"{CUR_OUT}/B4_semi_supervised", pct_tag=pct_tag,
)
del mm_model, optimizer; empty_cache()

In [ ]:
# ===== B5-full: SimCLR + MixMatch full FT =====
class SimCLRMixMatchFull(nn.Module):
    def __init__(self, pretrained_encoder, num_classes):
        super().__init__()
        self.encoder = copy.deepcopy(pretrained_encoder)
        self.classifier = nn.Linear(512, num_classes)
    def forward(self, x):
        return self.classifier(self.encoder(x))

set_seed(SEED)
simclr_clean = SimCLRModel(feature_dim=128, init="imagenet").to(device)
simclr_clean.load_state_dict(simclr_ckpt["model_state_dict"])

b5_full = SimCLRMixMatchFull(simclr_clean.encoder, NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(b5_full.parameters(), lr=1e-4)
EPOCHS = EPOCHS_DEFAULT

print("=" * 64); print(f"B5-full @ {pct_tag} (Aug-Naive): SimCLR + MixMatch full FT"); print("=" * 64)

b5_history = {"epoch": [], "loss_x": [], "loss_u": [], "val_acc": []}
best, best_state = 0.0, None
start = time.time()

for epoch in range(EPOCHS):
    b5_full.train()
    tlx, tlu, nb = 0.0, 0.0, 0
    uiter = iter(unlabeled_loader)
    for bx, by in labeled_loader:
        try: buv = next(uiter)
        except StopIteration:
            uiter = iter(unlabeled_loader); buv = next(uiter)
        bx, by = bx.to(device), by.to(device)
        buv = [v.to(device) for v in buv]
        with torch.no_grad():
            probs = torch.zeros(buv[0].size(0), NUM_CLASSES, device=device)
            for v in buv: probs += F.softmax(b5_full(v), dim=1)
            probs /= len(buv)
            pseudo = sharpen(probs, T=SHARPEN_TEMP)
        loh = F.one_hot(by, num_classes=NUM_CLASSES).float()
        ai = torch.cat([bx] + buv, dim=0)
        at = torch.cat([loh] + [pseudo] * len(buv), dim=0)
        si = torch.randperm(ai.size(0))
        mx, my = mixup(ai, at, ai[si], at[si], alpha=MIXUP_ALPHA)
        n = bx.size(0)
        xp, yp = mx[:n], my[:n]; up, uy = mx[n:], my[n:]
        lx = -torch.mean(torch.sum(yp * F.log_softmax(b5_full(xp), dim=1), dim=1))
        lu = F.mse_loss(F.softmax(b5_full(up), dim=1), uy)
        loss = lx + LAMBDA_U * lu
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tlx += lx.item(); tlu += lu.item(); nb += 1
    b5_full.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (b5_full(imgs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = 100 * correct / total
    b5_history["epoch"].append(epoch + 1)
    b5_history["loss_x"].append(tlx / nb); b5_history["loss_u"].append(tlu / nb)
    b5_history["val_acc"].append(val_acc)
    if val_acc > best:
        best = val_acc; best_state = copy.deepcopy(b5_full.state_dict())
    print(f"  Ep {epoch+1:>2}/{EPOCHS}  lx={tlx/nb:.3f}  lu={tlu/nb:.4f}  "
          f"val={val_acc:.2f}%  (best: {best:.2f}%)")

print(f"\nB5-full done. Time: {(time.time()-start)/60:.1f}m | Best: {best:.2f}%")
b5_full.load_state_dict(best_state)
results_cur["B5_hybrid"] = best
metrics_b5 = full_evaluation(
    model=b5_full, loader=val_loader, class_names=base_dataset.classes,
    method_name="B5_hybrid", history=b5_history,
    out_dir=f"{CUR_OUT}/B5_hybrid", pct_tag=pct_tag,
)
del b5_full, optimizer, simclr_clean; empty_cache()

print("\n" + "=" * 64); print(f"SUMMARY @ {pct_tag} (Aug-Naive)"); print("=" * 64)
for method, acc in results_cur.items():
    print(f"  {method:<25}  {acc:.2f}%")

In [ ]:
# ===== Section 13.5: Within-experiment comparison =====
import csv

METHOD_ORDER = ["B1_supervised", "B3_self_supervised", "B4_semi_supervised", "B5_hybrid"]
METHOD_DISPLAY = {
    "B1_supervised":      "B1: Supervised",
    "B3_self_supervised": "B3: SimCLR + FT",
    "B4_semi_supervised": "B4: MixMatch",
    "B5_hybrid":          "B5: SimCLR + MixMatch",
}

all_metrics = {}
for method in METHOD_ORDER:
    path = f"{CUR_OUT}/{method}/metrics.json"
    if os.path.exists(path):
        with open(path) as f:
            all_metrics[method] = json.load(f)

if len(all_metrics) < 4:
    print(f"Only {len(all_metrics)}/4 methods found — finish training first.")
else:
    class_names = base_dataset.classes
    n_classes  = len(class_names)
    n_methods  = len(METHOD_ORDER)
    save_dir   = f"{CUR_OUT}/_comparison"
    os.makedirs(save_dir, exist_ok=True)

    # Overall metrics figure
    metric_keys = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
    metric_labels = ["Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)"]
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(n_methods); w = 0.2
    for j, (mk, ml) in enumerate(zip(metric_keys, metric_labels)):
        vals = [all_metrics[m]["overall"][mk] for m in METHOD_ORDER]
        bars = ax.bar(x + (j - 1.5) * w, vals, w, label=ml)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width()/2, v + 0.005,
                    f"{v*100:.1f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([METHOD_DISPLAY[m] for m in METHOD_ORDER], rotation=15, ha="right")
    ax.set_ylim(0, 1.08); ax.set_ylabel("Score")
    ax.set_title(f"Augmented Taiwan (naive split) @ {pct_tag}")
    ax.legend(loc="lower right"); ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/overall_metrics_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Per-class F1 heatmap
    f1_matrix = np.zeros((n_methods, n_classes))
    for i, m in enumerate(METHOD_ORDER):
        f1_matrix[i] = all_metrics[m]["per_class"]["f1"]
    fig, ax = plt.subplots(figsize=(max(10, n_classes * 1.4), 5))
    im = ax.imshow(f1_matrix, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(n_classes))
    ax.set_xticklabels([c[:22] for c in class_names], rotation=30, ha="right")
    ax.set_yticks(range(n_methods))
    ax.set_yticklabels([METHOD_DISPLAY[m] for m in METHOD_ORDER])
    for i in range(n_methods):
        for j in range(n_classes):
            v = f1_matrix[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    color="white" if v < 0.5 else "black", fontsize=9)
    ax.set_title(f"Per-class F1 — Augmented Taiwan (naive split) @ {pct_tag}")
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01, label="F1 score")
    plt.tight_layout()
    plt.savefig(f"{save_dir}/per_class_f1_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Print + save comparison table
    print("\n" + "=" * 80)
    print(f"COMPARISON @ {pct_tag} (Augmented Taiwan, NAIVE split — leakage present)")
    print("=" * 80)
    print(f"  {'Method':<28} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}")
    print("  " + "-" * 62)
    rows = []
    for m in METHOD_ORDER:
        o = all_metrics[m]["overall"]
        rows.append([METHOD_DISPLAY[m], o["accuracy"]*100, o["precision_macro"]*100,
                     o["recall_macro"]*100, o["f1_macro"]*100])
        print(f"  {METHOD_DISPLAY[m]:<28} {o['accuracy']*100:>7.2f}% "
              f"{o['precision_macro']*100:>7.2f}% {o['recall_macro']*100:>7.2f}% "
              f"{o['f1_macro']*100:>7.2f}%")

    print("\n  --- For reference (TOM-SSL paper, augmented Taiwan, 10% labels) ---")
    print(f"  {'TOM-SSL (their paper)':<28} {'70.87':>7}% {'—':>7}  {'—':>7}  {'69.28':>7}%")
    print(f"  {'MixMatch (their paper)':<28} {'67.26':>7}% {'—':>7}  {'—':>7}  {'65.43':>7}%")
    print(f"  {'MobileNetV3-Sup (theirs)':<28} {'56.52':>7}% {'—':>7}  {'—':>7}  {'53.73':>7}%")

    with open(f"{save_dir}/comparison_table.csv", "w", newline="") as f:
        w_csv = csv.writer(f)
        w_csv.writerow(["Method","Accuracy","Precision_macro","Recall_macro","F1_macro","Protocol"])
        for r in rows:
            w_csv.writerow([r[0]] + [f"{x:.4f}" for x in r[1:]] + ["augmented_naive_split"])
        w_csv.writerow(["TOM-SSL", "0.7087", "—", "—", "0.6928", "augmented_naive_split"])
        w_csv.writerow(["MixMatch_tomssl", "0.6726", "—", "—", "0.6543", "augmented_naive_split"])

    print(f"\nSaved comparison to: {save_dir}")

In [ ]:
# ===== Source-grouped stratified split =====
# Guarantees no source image has copies in both labeled and validation splits.

import re

def source_id_from_filename(fname):
    """Extract source-image ID from augmented Taiwan filenames."""
    stem = os.path.splitext(fname)[0]
    m = re.match(r"^(Train|Test)_([A-Za-z]+\d+)", stem)
    if m:
        return f"{m.group(1)}_{m.group(2)}"
    return stem

def grouped_stratified_split(dataset, label_pct=0.10, val_pct=0.10,
                              min_per_class=10, seed=42):
    """
    Split at the SOURCE-IMAGE level, then assign all augmented copies
    of a source image to whichever split that source went to.

    This is what TOM-SSL should have done to avoid leakage.
    """
    rng = random.Random(seed)

    # Build: class -> {source_id: [dataset_indices]}
    class_to_sources = {}
    for idx in range(len(dataset)):
        path, label = dataset.samples[idx]
        src = source_id_from_filename(os.path.basename(path))
        class_to_sources.setdefault(label, {}).setdefault(src, []).append(idx)

    labeled_idx, unlabeled_idx, val_idx = [], [], []

    for label, src_dict in class_to_sources.items():
        src_ids = list(src_dict.keys())
        rng.shuffle(src_ids)

        # Convert per-class label/val targets from images to sources
        # (approximate: assume average copies/source ratio holds per class)
        total_imgs = sum(len(v) for v in src_dict.values())
        target_n_label = max(min_per_class, int(label_pct * total_imgs))
        target_n_val   = max(min_per_class, int(val_pct * total_imgs))

        # Pick sources for labeled split until we have enough images
        labeled_sources = []
        n_so_far = 0
        i = 0
        while n_so_far < target_n_label and i < len(src_ids):
            labeled_sources.append(src_ids[i])
            n_so_far += len(src_dict[src_ids[i]])
            i += 1

        # Pick sources for val split
        val_sources = []
        n_so_far = 0
        while n_so_far < target_n_val and i < len(src_ids):
            val_sources.append(src_ids[i])
            n_so_far += len(src_dict[src_ids[i]])
            i += 1

        # Remainder = unlabeled
        unlabeled_sources = src_ids[i:]

        # Flatten back to image indices
        for s in labeled_sources:
            labeled_idx.extend(src_dict[s])
        for s in val_sources:
            val_idx.extend(src_dict[s])
        for s in unlabeled_sources:
            unlabeled_idx.extend(src_dict[s])

    return labeled_idx, unlabeled_idx, val_idx

print("Grouped split function defined.")

In [ ]:
# ===== Experiment B: Augmented Taiwan + GROUPED (leak-free) split =====
RESULTS_AUG_GROUPED = f"{PROJECT_DIR}/RESULTS_TAIWAN_AUG_GROUPED"
for method in ["B1_supervised", "B3_self_supervised",
               "B4_semi_supervised", "B5_hybrid"]:
    os.makedirs(f"{RESULTS_AUG_GROUPED}/10pct/{method}", exist_ok=True)
os.makedirs(f"{RESULTS_AUG_GROUPED}/splits", exist_ok=True)

LABEL_PCT = 0.10
pct_tag = f"{int(LABEL_PCT*100)}pct"
CUR_OUT = f"{RESULTS_AUG_GROUPED}/{pct_tag}"
print(f"Experiment B (GROUPED split): LABEL_PCT={LABEL_PCT}")
print(f"Output base: {CUR_OUT}")

set_seed(SEED)
labeled_idx, unlabeled_idx, val_idx = grouped_stratified_split(
    base_dataset, label_pct=LABEL_PCT, val_pct=VAL_PCT,
    min_per_class=10, seed=SEED,
)

total_n = len(base_dataset)
print(f"\nSplit sizes (grouped):")
print(f"  Labeled:    {len(labeled_idx):>5} ({100*len(labeled_idx)/total_n:.2f}%)")
print(f"  Unlabeled:  {len(unlabeled_idx):>5} ({100*len(unlabeled_idx)/total_n:.2f}%)")
print(f"  Validation: {len(val_idx):>5} ({100*len(val_idx)/total_n:.2f}%)")

# Confirm: zero overlap
def src_of(idx):
    path, _ = base_dataset.samples[idx]
    return source_id_from_filename(os.path.basename(path))

labeled_sources = set(src_of(i) for i in labeled_idx)
val_sources     = set(src_of(i) for i in val_idx)
overlap = labeled_sources & val_sources
print(f"\n=== LEAKAGE DIAGNOSTIC (grouped split) ===")
print(f"  Distinct source IDs in labeled:    {len(labeled_sources)}")
print(f"  Distinct source IDs in val:        {len(val_sources)}")
print(f"  Source IDs appearing in BOTH:      {len(overlap)}")
print(f"  → Expect 0. Anything else = bug.")
assert len(overlap) == 0, "Grouped split has overlap — bug in grouping!"

with open(f"{RESULTS_AUG_GROUPED}/splits/split_{pct_tag}_seed{SEED}.json", "w") as f:
    json.dump({
        "seed": SEED, "label_pct": LABEL_PCT, "protocol": "source_grouped",
        "labeled_idx": labeled_idx, "unlabeled_idx": unlabeled_idx,
        "val_idx": val_idx,
        "n_labeled_sources": len(labeled_sources),
        "n_val_sources": len(val_sources),
        "classes": base_dataset.classes,
    }, f)

# Rebuild loaders for experiment B
labeled_dataset   = LabeledSubset(base_dataset, labeled_idx,   weak_transform)
unlabeled_dataset = UnlabeledKAug(base_dataset, unlabeled_idx, weak_transform, K=K_AUGMENTS)
val_dataset       = LabeledSubset(base_dataset, val_idx,       eval_transform)

labeled_loader = DataLoader(labeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
train_loader = DataLoader(labeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)

print(f"\nLoader batch counts:")
print(f"  labeled:   {len(labeled_loader)}")
print(f"  unlabeled: {len(unlabeled_loader)}")
print(f"  val:       {len(val_loader)}")
print(f"  train:     {len(train_loader)}")

results_cur = {}

In [ ]:
# ===== B1: Supervised (ImageNet init) — Grouped split =====
set_seed(SEED)
sup_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
sup_model.fc = nn.Linear(512, NUM_CLASSES)
sup_model = sup_model.to(device)

LR_SUP = 1e-3
optimizer = torch.optim.Adam(sup_model.parameters(), lr=LR_SUP)
criterion = nn.CrossEntropyLoss()
EPOCHS = EPOCHS_DEFAULT

print("=" * 64); print(f"B1 @ {pct_tag} (Aug-Grouped): Supervised (ImageNet init)"); print("=" * 64)

b1_history = {"epoch": [], "train_loss": [], "val_acc": []}
best, best_state = 0.0, None
start = time.time()

for epoch in range(EPOCHS):
    sup_model.train()
    tl, nb = 0.0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        loss = criterion(sup_model(imgs), labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tl += loss.item(); nb += 1
    sup_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (sup_model(imgs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = 100 * correct / total
    b1_history["epoch"].append(epoch + 1)
    b1_history["train_loss"].append(tl / nb)
    b1_history["val_acc"].append(val_acc)
    if val_acc > best:
        best = val_acc; best_state = copy.deepcopy(sup_model.state_dict())
    print(f"  Ep {epoch+1:>2}/{EPOCHS}  loss={tl/nb:.4f}  val={val_acc:.2f}%  (best: {best:.2f}%)")

print(f"\nB1 done. Time: {(time.time()-start)/60:.1f}m | Best: {best:.2f}%")
sup_model.load_state_dict(best_state)
results_cur["B1_supervised"] = best
metrics_b1 = full_evaluation(
    model=sup_model, loader=val_loader, class_names=base_dataset.classes,
    method_name="B1_supervised", history=b1_history,
    out_dir=f"{CUR_OUT}/B1_supervised", pct_tag=pct_tag,
)
del sup_model, optimizer; empty_cache()

In [ ]:
# ===== B3: SimCLR + Full Fine-tune — Grouped split =====
class SimCLRFineTune(nn.Module):
    def __init__(self, pretrained_encoder, num_classes):
        super().__init__()
        self.encoder = copy.deepcopy(pretrained_encoder)
        self.classifier = nn.Linear(512, num_classes)
    def forward(self, x):
        return self.classifier(self.encoder(x))

set_seed(SEED)
simclr_clean = SimCLRModel(feature_dim=128, init="imagenet").to(device)
simclr_clean.load_state_dict(simclr_ckpt["model_state_dict"])

ft_model = SimCLRFineTune(simclr_clean.encoder, NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(ft_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
EPOCHS = EPOCHS_DEFAULT

print("=" * 64); print(f"B3 @ {pct_tag} (Aug-Grouped): SimCLR + Full FT (LR=1e-4)"); print("=" * 64)

b3_history = {"epoch": [], "train_loss": [], "val_acc": []}
best, best_state = 0.0, None
start = time.time()

for epoch in range(EPOCHS):
    ft_model.train()
    tl, nb = 0.0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        loss = criterion(ft_model(imgs), labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tl += loss.item(); nb += 1
    ft_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (ft_model(imgs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = 100 * correct / total
    b3_history["epoch"].append(epoch + 1)
    b3_history["train_loss"].append(tl / nb)
    b3_history["val_acc"].append(val_acc)
    if val_acc > best:
        best = val_acc; best_state = copy.deepcopy(ft_model.state_dict())
    print(f"  Ep {epoch+1:>2}/{EPOCHS}  loss={tl/nb:.3f}  val={val_acc:.2f}%  (best: {best:.2f}%)")

print(f"\nB3 done. Time: {(time.time()-start)/60:.1f}m | Best: {best:.2f}%")
ft_model.load_state_dict(best_state)
results_cur["B3_self_supervised"] = best
metrics_b3 = full_evaluation(
    model=ft_model, loader=val_loader, class_names=base_dataset.classes,
    method_name="B3_self_supervised", history=b3_history,
    out_dir=f"{CUR_OUT}/B3_self_supervised", pct_tag=pct_tag,
)
del ft_model, optimizer, simclr_clean; empty_cache()

In [ ]:
# ===== B4: MixMatch (ImageNet init) — Grouped split =====
set_seed(SEED)
mm_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
mm_model.fc = nn.Linear(512, NUM_CLASSES)
mm_model = mm_model.to(device)

LR_MM = 1e-4
optimizer = torch.optim.Adam(mm_model.parameters(), lr=LR_MM)
EPOCHS = EPOCHS_DEFAULT

print("=" * 64); print(f"B4 @ {pct_tag} (Aug-Grouped): MixMatch (ImageNet init)"); print("=" * 64)

b4_history = {"epoch": [], "loss_x": [], "loss_u": [], "val_acc": []}
best, best_state = 0.0, None
start = time.time()

for epoch in range(EPOCHS):
    mm_model.train()
    tlx, tlu, nb = 0.0, 0.0, 0
    uiter = iter(unlabeled_loader)
    for bx, by in labeled_loader:
        try: buv = next(uiter)
        except StopIteration:
            uiter = iter(unlabeled_loader); buv = next(uiter)
        bx, by = bx.to(device), by.to(device)
        buv = [v.to(device) for v in buv]
        with torch.no_grad():
            probs = torch.zeros(buv[0].size(0), NUM_CLASSES, device=device)
            for v in buv: probs += F.softmax(mm_model(v), dim=1)
            probs /= len(buv)
            pseudo = sharpen(probs, T=SHARPEN_TEMP)
        loh = F.one_hot(by, num_classes=NUM_CLASSES).float()
        ai = torch.cat([bx] + buv, dim=0)
        at = torch.cat([loh] + [pseudo] * len(buv), dim=0)
        si = torch.randperm(ai.size(0))
        mx, my = mixup(ai, at, ai[si], at[si], alpha=MIXUP_ALPHA)
        n = bx.size(0)
        xp, yp = mx[:n], my[:n]; up, uy = mx[n:], my[n:]
        lx = -torch.mean(torch.sum(yp * F.log_softmax(mm_model(xp), dim=1), dim=1))
        lu = F.mse_loss(F.softmax(mm_model(up), dim=1), uy)
        loss = lx + LAMBDA_U * lu
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tlx += lx.item(); tlu += lu.item(); nb += 1
    mm_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (mm_model(imgs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = 100 * correct / total
    b4_history["epoch"].append(epoch + 1)
    b4_history["loss_x"].append(tlx / nb); b4_history["loss_u"].append(tlu / nb)
    b4_history["val_acc"].append(val_acc)
    if val_acc > best:
        best = val_acc; best_state = copy.deepcopy(mm_model.state_dict())
    print(f"  Ep {epoch+1:>2}/{EPOCHS}  lx={tlx/nb:.3f}  lu={tlu/nb:.4f}  "
          f"val={val_acc:.2f}%  (best: {best:.2f}%)")

print(f"\nB4 done. Time: {(time.time()-start)/60:.1f}m | Best: {best:.2f}%")
mm_model.load_state_dict(best_state)
results_cur["B4_semi_supervised"] = best
metrics_b4 = full_evaluation(
    model=mm_model, loader=val_loader, class_names=base_dataset.classes,
    method_name="B4_semi_supervised", history=b4_history,
    out_dir=f"{CUR_OUT}/B4_semi_supervised", pct_tag=pct_tag,
)
del mm_model, optimizer; empty_cache()

In [ ]:
# ===== B5-full: SimCLR + MixMatch full FT — Grouped split =====
class SimCLRMixMatchFull(nn.Module):
    def __init__(self, pretrained_encoder, num_classes):
        super().__init__()
        self.encoder = copy.deepcopy(pretrained_encoder)
        self.classifier = nn.Linear(512, num_classes)
    def forward(self, x):
        return self.classifier(self.encoder(x))

set_seed(SEED)
simclr_clean = SimCLRModel(feature_dim=128, init="imagenet").to(device)
simclr_clean.load_state_dict(simclr_ckpt["model_state_dict"])

b5_full = SimCLRMixMatchFull(simclr_clean.encoder, NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(b5_full.parameters(), lr=1e-4)
EPOCHS = EPOCHS_DEFAULT

print("=" * 64); print(f"B5-full @ {pct_tag} (Aug-Grouped): SimCLR + MixMatch full FT"); print("=" * 64)

b5_history = {"epoch": [], "loss_x": [], "loss_u": [], "val_acc": []}
best, best_state = 0.0, None
start = time.time()

for epoch in range(EPOCHS):
    b5_full.train()
    tlx, tlu, nb = 0.0, 0.0, 0
    uiter = iter(unlabeled_loader)
    for bx, by in labeled_loader:
        try: buv = next(uiter)
        except StopIteration:
            uiter = iter(unlabeled_loader); buv = next(uiter)
        bx, by = bx.to(device), by.to(device)
        buv = [v.to(device) for v in buv]
        with torch.no_grad():
            probs = torch.zeros(buv[0].size(0), NUM_CLASSES, device=device)
            for v in buv: probs += F.softmax(b5_full(v), dim=1)
            probs /= len(buv)
            pseudo = sharpen(probs, T=SHARPEN_TEMP)
        loh = F.one_hot(by, num_classes=NUM_CLASSES).float()
        ai = torch.cat([bx] + buv, dim=0)
        at = torch.cat([loh] + [pseudo] * len(buv), dim=0)
        si = torch.randperm(ai.size(0))
        mx, my = mixup(ai, at, ai[si], at[si], alpha=MIXUP_ALPHA)
        n = bx.size(0)
        xp, yp = mx[:n], my[:n]; up, uy = mx[n:], my[n:]
        lx = -torch.mean(torch.sum(yp * F.log_softmax(b5_full(xp), dim=1), dim=1))
        lu = F.mse_loss(F.softmax(b5_full(up), dim=1), uy)
        loss = lx + LAMBDA_U * lu
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tlx += lx.item(); tlu += lu.item(); nb += 1
    b5_full.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (b5_full(imgs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = 100 * correct / total
    b5_history["epoch"].append(epoch + 1)
    b5_history["loss_x"].append(tlx / nb); b5_history["loss_u"].append(tlu / nb)
    b5_history["val_acc"].append(val_acc)
    if val_acc > best:
        best = val_acc; best_state = copy.deepcopy(b5_full.state_dict())
    print(f"  Ep {epoch+1:>2}/{EPOCHS}  lx={tlx/nb:.3f}  lu={tlu/nb:.4f}  "
          f"val={val_acc:.2f}%  (best: {best:.2f}%)")

print(f"\nB5-full done. Time: {(time.time()-start)/60:.1f}m | Best: {best:.2f}%")
b5_full.load_state_dict(best_state)
results_cur["B5_hybrid"] = best
metrics_b5 = full_evaluation(
    model=b5_full, loader=val_loader, class_names=base_dataset.classes,
    method_name="B5_hybrid", history=b5_history,
    out_dir=f"{CUR_OUT}/B5_hybrid", pct_tag=pct_tag,
)
del b5_full, optimizer, simclr_clean; empty_cache()

print("\n" + "=" * 64); print(f"SUMMARY @ {pct_tag} (Aug-Grouped)"); print("=" * 64)
for method, acc in results_cur.items():
    print(f"  {method:<25}  {acc:.2f}%")

In [ ]:
# ===== Within-experiment comparison (Aug-Grouped) =====
import csv

METHOD_ORDER = ["B1_supervised", "B3_self_supervised", "B4_semi_supervised", "B5_hybrid"]
METHOD_DISPLAY = {
    "B1_supervised":      "B1: Supervised",
    "B3_self_supervised": "B3: SimCLR + FT",
    "B4_semi_supervised": "B4: MixMatch",
    "B5_hybrid":          "B5: SimCLR + MixMatch",
}

all_metrics = {}
for method in METHOD_ORDER:
    path = f"{CUR_OUT}/{method}/metrics.json"
    if os.path.exists(path):
        with open(path) as f:
            all_metrics[method] = json.load(f)

if len(all_metrics) < 4:
    print(f"Only {len(all_metrics)}/4 methods found — finish training first.")
else:
    class_names = base_dataset.classes
    n_classes  = len(class_names)
    n_methods  = len(METHOD_ORDER)
    save_dir   = f"{CUR_OUT}/_comparison"
    os.makedirs(save_dir, exist_ok=True)

    # Overall metrics figure
    metric_keys = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
    metric_labels = ["Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)"]
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(n_methods); w = 0.2
    for j, (mk, ml) in enumerate(zip(metric_keys, metric_labels)):
        vals = [all_metrics[m]["overall"][mk] for m in METHOD_ORDER]
        bars = ax.bar(x + (j - 1.5) * w, vals, w, label=ml)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width()/2, v + 0.005,
                    f"{v*100:.1f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([METHOD_DISPLAY[m] for m in METHOD_ORDER], rotation=15, ha="right")
    ax.set_ylim(0, 1.08); ax.set_ylabel("Score")
    ax.set_title(f"Augmented Taiwan (GROUPED split, leak-free) @ {pct_tag}")
    ax.legend(loc="lower right"); ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/overall_metrics_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Per-class F1 heatmap
    f1_matrix = np.zeros((n_methods, n_classes))
    for i, m in enumerate(METHOD_ORDER):
        f1_matrix[i] = all_metrics[m]["per_class"]["f1"]
    fig, ax = plt.subplots(figsize=(max(10, n_classes * 1.4), 5))
    im = ax.imshow(f1_matrix, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(n_classes))
    ax.set_xticklabels([c[:22] for c in class_names], rotation=30, ha="right")
    ax.set_yticks(range(n_methods))
    ax.set_yticklabels([METHOD_DISPLAY[m] for m in METHOD_ORDER])
    for i in range(n_methods):
        for j in range(n_classes):
            v = f1_matrix[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    color="white" if v < 0.5 else "black", fontsize=9)
    ax.set_title(f"Per-class F1 — Augmented Taiwan GROUPED @ {pct_tag}")
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01, label="F1 score")
    plt.tight_layout()
    plt.savefig(f"{save_dir}/per_class_f1_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\n" + "=" * 80)
    print(f"COMPARISON @ {pct_tag} (Augmented Taiwan, GROUPED split — leak-free)")
    print("=" * 80)
    print(f"  {'Method':<28} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}")
    print("  " + "-" * 62)
    rows = []
    for m in METHOD_ORDER:
        o = all_metrics[m]["overall"]
        rows.append([METHOD_DISPLAY[m], o["accuracy"]*100, o["precision_macro"]*100,
                     o["recall_macro"]*100, o["f1_macro"]*100])
        print(f"  {METHOD_DISPLAY[m]:<28} {o['accuracy']*100:>7.2f}% "
              f"{o['precision_macro']*100:>7.2f}% {o['recall_macro']*100:>7.2f}% "
              f"{o['f1_macro']*100:>7.2f}%")

    with open(f"{save_dir}/comparison_table.csv", "w", newline="") as f:
        w_csv = csv.writer(f)
        w_csv.writerow(["Method","Accuracy","Precision_macro","Recall_macro","F1_macro","Protocol"])
        for r in rows:
            w_csv.writerow([r[0]] + [f"{x:.4f}" for x in r[1:]] + ["augmented_grouped_split"])

    print(f"\nSaved comparison to: {save_dir}")

In [ ]:
# ===== MASTER CROSS-PROTOCOL COMPARISON (Taiwan) =====
import csv
import matplotlib.patches as mpatches

METHOD_ORDER = ["B1_supervised", "B3_self_supervised", "B4_semi_supervised", "B5_hybrid"]
METHOD_DISPLAY = {
    "B1_supervised":      "B1: Supervised",
    "B3_self_supervised": "B3: SimCLR + FT",
    "B4_semi_supervised": "B4: MixMatch",
    "B5_hybrid":          "B5: SimCLR + MixMatch",
}

PROTOCOLS = {
    "preprocessed":  ("Original (622 imgs)",          f"{PROJECT_DIR}/RESULTS_TAIWAN/10pct"),
    "aug_naive":     ("Augmented + naive (TOM-SSL)",  f"{PROJECT_DIR}/RESULTS_TAIWAN_AUG_NAIVE/10pct"),
    "aug_grouped":   ("Augmented + grouped (leak-free)", f"{PROJECT_DIR}/RESULTS_TAIWAN_AUG_GROUPED/10pct"),
}

# Load all metrics
all_results = {}
for proto_key, (proto_label, proto_dir) in PROTOCOLS.items():
    all_results[proto_key] = {}
    for method in METHOD_ORDER:
        path = f"{proto_dir}/{method}/metrics.json"
        if os.path.exists(path):
            with open(path) as f:
                all_results[proto_key][method] = json.load(f)

# === Headline table ===
print("\n" + "=" * 92)
print("MASTER COMPARISON — Taiwan dataset @ 10% labels (seed 42)")
print("=" * 92)
header = f"  {'Method':<25}" + "".join(f" {PROTOCOLS[k][0][:24]:>26}" for k in PROTOCOLS)
print(header)
print("  " + "-" * 90)
for m in METHOD_ORDER:
    row = f"  {METHOD_DISPLAY[m]:<25}"
    for proto_key in PROTOCOLS:
        if m in all_results[proto_key]:
            acc = all_results[proto_key][m]["overall"]["accuracy"] * 100
            f1  = all_results[proto_key][m]["overall"]["f1_macro"] * 100
            row += f"  Acc {acc:>5.2f}% F1 {f1:>5.2f}%  "
        else:
            row += f"  {'—':>26}"
    print(row)

print("\n  --- Reference (TOM-SSL paper) ---")
print(f"  {'TOM-SSL (reported)':<25}                                 Acc {70.87:>5.2f}% F1 {69.28:>5.2f}%")
print(f"  {'MixMatch (their paper)':<25}                                 Acc {67.26:>5.2f}% F1 {65.43:>5.2f}%")
print(f"  {'MobileNetV3-Sup (theirs)':<25}                                 Acc {56.52:>5.2f}% F1 {53.73:>5.2f}%")

# === Leakage effect quantification ===
print("\n" + "=" * 92)
print("LEAKAGE EFFECT (Naive minus Grouped on the same augmented data)")
print("=" * 92)
print(f"  {'Method':<25}  {'Naive Acc':>12}  {'Grouped Acc':>12}  {'Δ (leakage)':>12}")
print("  " + "-" * 70)
for m in METHOD_ORDER:
    naive_acc   = all_results["aug_naive"][m]["overall"]["accuracy"] * 100
    grouped_acc = all_results["aug_grouped"][m]["overall"]["accuracy"] * 100
    delta = naive_acc - grouped_acc
    print(f"  {METHOD_DISPLAY[m]:<25}  {naive_acc:>11.2f}%  {grouped_acc:>11.2f}%  {delta:>+11.2f}")
mean_delta = np.mean([
    all_results["aug_naive"][m]["overall"]["accuracy"] -
    all_results["aug_grouped"][m]["overall"]["accuracy"]
    for m in METHOD_ORDER
]) * 100
print(f"\n  Mean leakage inflation: +{mean_delta:.2f} accuracy points")

# === Save master CSV ===
SUMMARY_DIR = f"{PROJECT_DIR}/RESULTS_TAIWAN_MASTER"
os.makedirs(SUMMARY_DIR, exist_ok=True)
with open(f"{SUMMARY_DIR}/master_comparison.csv", "w", newline="") as f:
    w_csv = csv.writer(f)
    w_csv.writerow(["Method", "Protocol", "Accuracy", "Precision_macro",
                    "Recall_macro", "F1_macro"])
    for proto_key, (proto_label, _) in PROTOCOLS.items():
        for m in METHOD_ORDER:
            if m in all_results[proto_key]:
                o = all_results[proto_key][m]["overall"]
                w_csv.writerow([METHOD_DISPLAY[m], proto_label,
                                f"{o['accuracy']:.4f}", f"{o['precision_macro']:.4f}",
                                f"{o['recall_macro']:.4f}", f"{o['f1_macro']:.4f}"])
    # Add TOM-SSL benchmarks
    w_csv.writerow(["TOM-SSL (reported)",       "Augmented + naive (TOM-SSL)", "0.7087", "—", "—", "0.6928"])
    w_csv.writerow(["MixMatch (their paper)",   "Augmented + naive (TOM-SSL)", "0.6726", "—", "—", "0.6543"])
    w_csv.writerow(["MobileNetV3-Sup (theirs)", "Augmented + naive (TOM-SSL)", "0.5652", "—", "—", "0.5373"])

# === Figure 1: 3-protocol grouped bars (accuracy) ===
fig, ax = plt.subplots(figsize=(13, 6))
n_methods = len(METHOD_ORDER); n_protos = len(PROTOCOLS)
x = np.arange(n_methods); w = 0.25
colors = ["#999999", "#E69F00", "#56B4E9"]  # gray=preprocessed, orange=naive, blue=grouped

for i, (proto_key, (proto_label, _)) in enumerate(PROTOCOLS.items()):
    vals = [all_results[proto_key][m]["overall"]["accuracy"] for m in METHOD_ORDER
            if m in all_results[proto_key]]
    bars = ax.bar(x + (i - 1) * w, vals, w, label=proto_label, color=colors[i])
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + 0.005,
                f"{v*100:.1f}", ha="center", va="bottom", fontsize=8)

# TOM-SSL reference line
ax.axhline(0.7087, ls="--", color="red", alpha=0.5, label="TOM-SSL reported (70.87%)")
ax.set_xticks(x)
ax.set_xticklabels([METHOD_DISPLAY[m] for m in METHOD_ORDER], rotation=15, ha="right")
ax.set_ylim(0, 1.02); ax.set_ylabel("Accuracy")
ax.set_title("Taiwan dataset @ 10% labels — three protocols")
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SUMMARY_DIR}/master_accuracy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# === Figure 2: Leakage effect (paired bars naive vs grouped, with delta annotated) ===
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(n_methods); w = 0.35

naive_vals   = [all_results["aug_naive"][m]["overall"]["accuracy"] for m in METHOD_ORDER]
grouped_vals = [all_results["aug_grouped"][m]["overall"]["accuracy"] for m in METHOD_ORDER]

b1 = ax.bar(x - w/2, naive_vals,   w, label="Naive split (leakage)",  color="#E69F00")
b2 = ax.bar(x + w/2, grouped_vals, w, label="Grouped split (leak-free)", color="#56B4E9")

for bar_n, bar_g, n, g in zip(b1, b2, naive_vals, grouped_vals):
    ax.text(bar_n.get_x() + bar_n.get_width()/2, n + 0.005,
            f"{n*100:.1f}", ha="center", va="bottom", fontsize=8)
    ax.text(bar_g.get_x() + bar_g.get_width()/2, g + 0.005,
            f"{g*100:.1f}", ha="center", va="bottom", fontsize=8)
    # Delta annotation
    mid_x = (bar_n.get_x() + bar_g.get_x() + bar_g.get_width()) / 2
    delta = (n - g) * 100
    ax.annotate(f"Δ −{delta:.1f}", xy=(mid_x, max(n, g) + 0.05),
                ha="center", fontsize=9, color="red", fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels([METHOD_DISPLAY[m] for m in METHOD_ORDER], rotation=15, ha="right")
ax.set_ylim(0, 1.05); ax.set_ylabel("Accuracy")
ax.set_title("Augmented Taiwan @ 10% labels — leakage inflation by method")
ax.legend(loc="lower right")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SUMMARY_DIR}/master_leakage_effect.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nMaster comparison saved to: {SUMMARY_DIR}")

In [71]:
# ===== Recovery: Mount Drive, imports, device =====
from google.colab import drive
drive.mount('/content/drive')

import os, sys, random, copy, time, json, re, shutil
import numpy as np
import matplotlib.pyplot as plt
import torch, torchvision
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch: {torch.__version__}  Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch: 2.11.0+cu128  Device: cuda
GPU: Tesla T4


In [78]:
# ===== Recovery: Paths and hyperparameters =====
PROJECT_DIR = "/content/drive/MyDrive/Tomato_3regime"
RESULTS_DIR = f"{PROJECT_DIR}/RESULTS 2"
SIMCLR_CKPT_SRC = f"{RESULTS_DIR}/simclr/simclr_checkpoint.pth"

# Hyperparameters (same as before)
VAL_PCT     = 0.10
IMG_SIZE    = 224
BATCH_SIZE  = 128
NUM_WORKERS = 2
SIMCLR_TEMP = 0.5
K_AUGMENTS  = 2
SHARPEN_TEMP = 0.5
MIXUP_ALPHA  = 0.75
LAMBDA_U     = 1.0
EPOCHS_DEFAULT = 30
SEED = 42  # default; per-seed loop will override

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def empty_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"Project: {PROJECT_DIR}")
print(f"SimCLR checkpoint exists: {os.path.exists(SIMCLR_CKPT_SRC)}")

Project: /content/drive/MyDrive/Tomato_3regime
SimCLR checkpoint exists: True


In [79]:
# ===== Recovery: Rebuild Taiwan_aug_flat on local disk =====
AUG_SRC  = "/content/drive/MyDrive/taiwan/data augmentation"
AUG_FLAT = "/content/Taiwan_aug_flat"

if os.path.exists(AUG_FLAT):
    print(f"Already exists at {AUG_FLAT} — skipping rebuild.")
else:
    print(f"Rebuilding {AUG_FLAT} from Drive...")
    os.makedirs(AUG_FLAT, exist_ok=True)
    for split in ["Train", "Test"]:
        split_path = os.path.join(AUG_SRC, split)
        if not os.path.isdir(split_path):
            continue
        for cls in os.listdir(split_path):
            cls_src = os.path.join(split_path, cls)
            if not os.path.isdir(cls_src):
                continue
            cls_dst = os.path.join(AUG_FLAT, cls)
            os.makedirs(cls_dst, exist_ok=True)
            for fname in os.listdir(cls_src):
                shutil.copy(os.path.join(cls_src, fname),
                            os.path.join(cls_dst, f"{split}_{fname}"))

# Verify
total = 0
for cls in sorted(os.listdir(AUG_FLAT)):
    n = len(os.listdir(os.path.join(AUG_FLAT, cls)))
    total += n
    print(f"  {cls:<25} {n:>5}")
print(f"  {'TOTAL':<25} {total:>5}")
assert total == 4976, f"Expected 4976, got {total}"
print(f"\n✓ Augmented dataset ready: {AUG_FLAT}")

Already exists at /content/Taiwan_aug_flat — skipping rebuild.
  Bacterial spot              880
  Black mold                  536
  Gray spot                   672
  Late blight                 784
  health                      848
  powdery mildew             1256
  TOTAL                      4976

✓ Augmented dataset ready: /content/Taiwan_aug_flat


In [80]:
# ===== Recovery: Transforms and dataset wrappers =====
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(IMG_SIZE, padding=16),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
weak_transform = train_transform

class LabeledSubset(Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base = base_dataset; self.indices = indices; self.transform = transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        path, label = self.base.samples[self.indices[i]]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label

class UnlabeledKAug(Dataset):
    def __init__(self, base_dataset, indices, transform, K=2):
        self.base = base_dataset; self.indices = indices
        self.transform = transform; self.K = K
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        path, _ = self.base.samples[self.indices[i]]
        img = Image.open(path).convert("RGB")
        return [self.transform(img) for _ in range(self.K)]

def sharpen(p, T=0.5):
    p = p.pow(1.0 / T)
    return p / p.sum(dim=1, keepdim=True)

def mixup(x1, y1, x2, y2, alpha=0.75):
    lam = np.random.beta(alpha, alpha); lam = max(lam, 1 - lam)
    return lam * x1 + (1 - lam) * x2, lam * y1 + (1 - lam) * y2

print("Transforms and dataset wrappers ready.")

Transforms and dataset wrappers ready.


In [81]:
# ===== Recovery: Base dataset + both split functions =====
base_dataset = ImageFolder(root=AUG_FLAT)
NUM_CLASSES = len(base_dataset.classes)
print(f"Classes ({NUM_CLASSES}): {base_dataset.classes}")
print(f"Total images: {len(base_dataset)}")


def stratified_split_with_floor(dataset, label_pct=0.10, val_pct=0.10,
                                 min_per_class=10, seed=42):
    rng = random.Random(seed)
    class_indices = {}
    for idx in range(len(dataset)):
        _, label = dataset.samples[idx]
        class_indices.setdefault(label, []).append(idx)
    labeled_idx, unlabeled_idx, val_idx = [], [], []
    for label, indices in class_indices.items():
        rng.shuffle(indices)
        n = len(indices)
        n_label = max(min_per_class, int(label_pct * n))
        n_val   = max(min_per_class, int(val_pct * n))
        labeled_idx.extend(indices[:n_label])
        val_idx.extend(indices[n_label:n_label + n_val])
        unlabeled_idx.extend(indices[n_label + n_val:])
    return labeled_idx, unlabeled_idx, val_idx


def source_id_from_filename(fname):
    stem = os.path.splitext(fname)[0]
    m = re.match(r"^(Train|Test)_([A-Za-z]+\d+)", stem)
    if m:
        return f"{m.group(1)}_{m.group(2)}"
    return stem


def grouped_stratified_split(dataset, label_pct=0.10, val_pct=0.10,
                              min_per_class=10, seed=42):
    rng = random.Random(seed)
    class_to_sources = {}
    for idx in range(len(dataset)):
        path, label = dataset.samples[idx]
        src = source_id_from_filename(os.path.basename(path))
        class_to_sources.setdefault(label, {}).setdefault(src, []).append(idx)

    labeled_idx, unlabeled_idx, val_idx = [], [], []
    for label, src_dict in class_to_sources.items():
        src_ids = list(src_dict.keys())
        rng.shuffle(src_ids)
        total_imgs = sum(len(v) for v in src_dict.values())
        target_n_label = max(min_per_class, int(label_pct * total_imgs))
        target_n_val   = max(min_per_class, int(val_pct * total_imgs))

        labeled_sources, n_so_far, i = [], 0, 0
        while n_so_far < target_n_label and i < len(src_ids):
            labeled_sources.append(src_ids[i])
            n_so_far += len(src_dict[src_ids[i]]); i += 1
        val_sources, n_so_far = [], 0
        while n_so_far < target_n_val and i < len(src_ids):
            val_sources.append(src_ids[i])
            n_so_far += len(src_dict[src_ids[i]]); i += 1
        unlabeled_sources = src_ids[i:]

        for s in labeled_sources:   labeled_idx.extend(src_dict[s])
        for s in val_sources:       val_idx.extend(src_dict[s])
        for s in unlabeled_sources: unlabeled_idx.extend(src_dict[s])
    return labeled_idx, unlabeled_idx, val_idx

print("Split functions defined.")

Classes (6): ['Bacterial spot', 'Black mold', 'Gray spot', 'Late blight', 'health', 'powdery mildew']
Total images: 4976
Split functions defined.


In [82]:
# ===== Recovery: Metrics + evaluation utilities =====

@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        y_pred.append(logits.argmax(1).cpu().numpy())
        y_true.append(labels.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)


def plot_confusion_matrix(cm, class_names, save_path, title="Confusion Matrix"):
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    for ax, mat, sub, fmt in [(axes[0], cm, "Counts", "d"),
                              (axes[1], cm_norm, "Normalized", ".2f")]:
        im = ax.imshow(mat, cmap="Blues", aspect="auto")
        ax.set_title(f"{title} — {sub}")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names)))
        ax.set_xticklabels([c[:20] for c in class_names], rotation=45, ha="right")
        ax.set_yticklabels([c[:20] for c in class_names])
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = mat[i, j]
                color = "white" if val > mat.max() * 0.55 else "black"
                ax.text(j, i, format(val, fmt), ha="center", va="center",
                        color=color, fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.close(fig)


def plot_training_curves(history, save_path, method_name):
    epochs = history.get("epoch", [])
    has_lx = "loss_x" in history
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    if has_lx:
        axes[0].plot(epochs, history["loss_x"], marker="o", label="labeled (lx)")
        axes[0].plot(epochs, history["loss_u"], marker="s", label="unlabeled (lu)")
    else:
        axes[0].plot(epochs, history["train_loss"], marker="o", label="train loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title(f"{method_name} — training loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(epochs, history["val_acc"], marker="o", color="tab:green")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Val accuracy (%)")
    axes[1].set_title(f"{method_name} — validation accuracy")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.close(fig)


def plot_per_class_bars(per_class, class_names, save_path, method_name):
    n = len(class_names); x = np.arange(n); w = 0.2
    fig, ax = plt.subplots(figsize=(max(10, n * 1.4), 6))
    ax.bar(x - 1.5*w, per_class["accuracy"],  w, label="Accuracy")
    ax.bar(x - 0.5*w, per_class["precision"], w, label="Precision")
    ax.bar(x + 0.5*w, per_class["recall"],    w, label="Recall")
    ax.bar(x + 1.5*w, per_class["f1"],        w, label="F1")
    ax.set_xticks(x)
    ax.set_xticklabels([c[:22] for c in class_names], rotation=30, ha="right")
    ax.set_ylim(0, 1.05); ax.set_ylabel("Score")
    ax.set_title(f"{method_name} — per-class metrics")
    ax.legend(loc="lower right"); ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.close(fig)


def full_evaluation(model, loader, class_names, method_name, history,
                    out_dir, pct_tag, model_state=None):
    os.makedirs(out_dir, exist_ok=True)
    y_true, y_pred = collect_predictions(model, loader, device)
    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    p_weight, r_weight, f1_weight, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0)
    p_cls, r_cls, f1_cls, support = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(class_names)), zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    row_sum = cm.sum(axis=1).clip(min=1)
    acc_cls = cm.diagonal() / row_sum
    per_class = {
        "class_names": list(class_names),
        "accuracy": acc_cls.tolist(), "precision": p_cls.tolist(),
        "recall": r_cls.tolist(), "f1": f1_cls.tolist(),
        "support": support.tolist(),
    }
    metrics = {
        "method": method_name, "pct_tag": pct_tag, "seed": SEED,
        "overall": {
            "accuracy": float(acc),
            "precision_macro": float(p_macro), "recall_macro": float(r_macro),
            "f1_macro": float(f1_macro),
            "precision_weighted": float(p_weight), "recall_weighted": float(r_weight),
            "f1_weighted": float(f1_weight),
        },
        "per_class": per_class, "confusion_matrix": cm.tolist(),
        "history": history,
        "best_val_acc": max(history.get("val_acc", [0.0])),
    }
    with open(f"{out_dir}/metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    report = classification_report(
        y_true, y_pred, labels=range(len(class_names)),
        target_names=class_names, digits=4, zero_division=0)
    with open(f"{out_dir}/classification_report.txt", "w") as f:
        f.write(f"Method: {method_name}   Label %: {pct_tag}   Seed: {SEED}\n")
        f.write("=" * 80 + "\n"); f.write(report)
        f.write("\n\nConfusion matrix (rows=true, cols=pred):\n"); f.write(str(cm))
    plot_confusion_matrix(cm, class_names,
        f"{out_dir}/confusion_matrix.png", title=f"{method_name} @ {pct_tag}")
    plot_training_curves(history,
        f"{out_dir}/training_curves.png", f"{method_name} @ {pct_tag}")
    plot_per_class_bars(per_class, class_names,
        f"{out_dir}/per_class_metrics.png", method_name=f"{method_name} @ {pct_tag}")
    state = model_state if model_state is not None else model.state_dict()
    torch.save({"state_dict": state, "method": method_name,
                "pct_tag": pct_tag, "history": history},
               f"{out_dir}/checkpoint.pth")
    return metrics

print("full_evaluation() ready.")

full_evaluation() ready.


In [83]:
# ===== Recovery: SimCLR model class + load checkpoint =====
class SimCLRModel(nn.Module):
    def __init__(self, feature_dim=128, init="imagenet"):
        super().__init__()
        if init == "imagenet":
            weights = models.ResNet18_Weights.IMAGENET1K_V1
        elif init == "random":
            weights = None
        else:
            raise ValueError(f"Unknown init: {init}")
        self.encoder = models.resnet18(weights=weights)
        self.encoder.fc = nn.Identity()
        self.projector = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Linear(256, feature_dim),
        )
    def forward(self, x):
        h = self.encoder(x); z = self.projector(h)
        return h, z

simclr_ckpt = torch.load(SIMCLR_CKPT_SRC, map_location=device, weights_only=False)
print(f"SimCLR checkpoint loaded (final loss = {simclr_ckpt['final_loss']:.4f})")
print(f"\n✓ Environment fully restored. You can now run the multi-seed sweep cells.")

SimCLR checkpoint loaded (final loss = 3.7202)

✓ Environment fully restored. You can now run the multi-seed sweep cells.


In [84]:
# ==============================================================================
# MULTI-SEED SWEEP — Aug-Naive (Augmented Taiwan + naive split)
# Seeds 123 and 456. Seed 42 already done — outputs preserved in 10pct/.
# ==============================================================================
SEEDS_TO_RUN = [123, 456]

# Make sure we're pointing at the augmented dataset
base_dataset = ImageFolder(root=AUG_FLAT)
NUM_CLASSES  = len(base_dataset.classes)
print(f"Multi-seed sweep on Aug-Naive ({NUM_CLASSES} classes, {len(base_dataset)} images)")
print(f"Seeds: {SEEDS_TO_RUN}\n")

overall_start = time.time()

for SEED_CUR in SEEDS_TO_RUN:
    print("\n" + "█" * 80)
    print(f"█  AUG-NAIVE  ·  SEED {SEED_CUR}")
    print("█" * 80)

    # Per-seed output folder
    CUR_OUT = f"{PROJECT_DIR}/RESULTS_TAIWAN_AUG_NAIVE/10pct_seed{SEED_CUR}"
    for method in ["B1_supervised", "B3_self_supervised",
                   "B4_semi_supervised", "B5_hybrid"]:
        os.makedirs(f"{CUR_OUT}/{method}", exist_ok=True)

    # ----- Build naive split with this seed -----
    set_seed(SEED_CUR)
    labeled_idx, unlabeled_idx, val_idx = stratified_split_with_floor(
        base_dataset, label_pct=0.10, val_pct=0.10, min_per_class=10, seed=SEED_CUR,
    )

    # Quick leakage diagnostic (informational)
    def src_of(idx):
        path, _ = base_dataset.samples[idx]
        return source_id_from_filename(os.path.basename(path))
    overlap = set(src_of(i) for i in labeled_idx) & set(src_of(i) for i in val_idx)
    val_srcs = set(src_of(i) for i in val_idx)
    print(f"  Split: lab={len(labeled_idx)}  unl={len(unlabeled_idx)}  val={len(val_idx)}")
    print(f"  Leakage: {len(overlap)} of {len(val_srcs)} val sources have a sibling in train "
          f"({100*len(overlap)/len(val_srcs):.1f}%)\n")

    # ----- Loaders -----
    labeled_dataset   = LabeledSubset(base_dataset, labeled_idx,   weak_transform)
    unlabeled_dataset = UnlabeledKAug(base_dataset, unlabeled_idx, weak_transform, K=K_AUGMENTS)
    val_dataset       = LabeledSubset(base_dataset, val_idx,       eval_transform)
    labeled_loader = DataLoader(labeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
    unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    train_loader = DataLoader(labeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)

    # ----- B1: Supervised -----
    set_seed(SEED_CUR)
    sup_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    sup_model.fc = nn.Linear(512, NUM_CLASSES); sup_model = sup_model.to(device)
    optimizer = torch.optim.Adam(sup_model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    hist = {"epoch": [], "train_loss": [], "val_acc": []}
    best, best_state = 0.0, None
    print(f"  Training B1 (seed {SEED_CUR})...", end="", flush=True)
    t0 = time.time()
    for epoch in range(EPOCHS_DEFAULT):
        sup_model.train(); tl, nb = 0.0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            loss = criterion(sup_model(imgs), labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            tl += loss.item(); nb += 1
        sup_model.eval(); correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                correct += (sup_model(imgs).argmax(1) == labels).sum().item()
                total += labels.size(0)
        val_acc = 100 * correct / total
        hist["epoch"].append(epoch+1); hist["train_loss"].append(tl/nb); hist["val_acc"].append(val_acc)
        if val_acc > best: best = val_acc; best_state = copy.deepcopy(sup_model.state_dict())
    sup_model.load_state_dict(best_state)
    # Need to override SEED in metrics — full_evaluation reads global SEED, so we patch it
    _saved_SEED = SEED; globals()["SEED"] = SEED_CUR
    full_evaluation(sup_model, val_loader, base_dataset.classes, "B1_supervised",
                    hist, f"{CUR_OUT}/B1_supervised", "10pct")
    globals()["SEED"] = _saved_SEED
    print(f" best={best:.2f}%  ({(time.time()-t0)/60:.1f}m)")
    del sup_model, optimizer; empty_cache()

    # ----- B3: SimCLR + Full FT -----
    set_seed(SEED_CUR)
    simclr_clean = SimCLRModel(feature_dim=128, init="imagenet").to(device)
    simclr_clean.load_state_dict(simclr_ckpt["model_state_dict"])
    class SimCLRFineTune(nn.Module):
        def __init__(self, enc, n): super().__init__(); self.encoder = copy.deepcopy(enc); self.classifier = nn.Linear(512, n)
        def forward(self, x): return self.classifier(self.encoder(x))
    ft_model = SimCLRFineTune(simclr_clean.encoder, NUM_CLASSES).to(device)
    optimizer = torch.optim.Adam(ft_model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    hist = {"epoch": [], "train_loss": [], "val_acc": []}
    best, best_state = 0.0, None
    print(f"  Training B3 (seed {SEED_CUR})...", end="", flush=True)
    t0 = time.time()
    for epoch in range(EPOCHS_DEFAULT):
        ft_model.train(); tl, nb = 0.0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            loss = criterion(ft_model(imgs), labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            tl += loss.item(); nb += 1
        ft_model.eval(); correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                correct += (ft_model(imgs).argmax(1) == labels).sum().item()
                total += labels.size(0)
        val_acc = 100 * correct / total
        hist["epoch"].append(epoch+1); hist["train_loss"].append(tl/nb); hist["val_acc"].append(val_acc)
        if val_acc > best: best = val_acc; best_state = copy.deepcopy(ft_model.state_dict())
    ft_model.load_state_dict(best_state)
    _saved_SEED = SEED; globals()["SEED"] = SEED_CUR
    full_evaluation(ft_model, val_loader, base_dataset.classes, "B3_self_supervised",
                    hist, f"{CUR_OUT}/B3_self_supervised", "10pct")
    globals()["SEED"] = _saved_SEED
    print(f" best={best:.2f}%  ({(time.time()-t0)/60:.1f}m)")
    del ft_model, optimizer, simclr_clean; empty_cache()

    # ----- B4: MixMatch -----
    set_seed(SEED_CUR)
    mm_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    mm_model.fc = nn.Linear(512, NUM_CLASSES); mm_model = mm_model.to(device)
    optimizer = torch.optim.Adam(mm_model.parameters(), lr=1e-4)
    hist = {"epoch": [], "loss_x": [], "loss_u": [], "val_acc": []}
    best, best_state = 0.0, None
    print(f"  Training B4 (seed {SEED_CUR})...", end="", flush=True)
    t0 = time.time()
    for epoch in range(EPOCHS_DEFAULT):
        mm_model.train(); tlx, tlu, nb = 0.0, 0.0, 0
        uiter = iter(unlabeled_loader)
        for bx, by in labeled_loader:
            try: buv = next(uiter)
            except StopIteration: uiter = iter(unlabeled_loader); buv = next(uiter)
            bx, by = bx.to(device), by.to(device); buv = [v.to(device) for v in buv]
            with torch.no_grad():
                probs = torch.zeros(buv[0].size(0), NUM_CLASSES, device=device)
                for v in buv: probs += F.softmax(mm_model(v), dim=1)
                probs /= len(buv); pseudo = sharpen(probs, T=SHARPEN_TEMP)
            loh = F.one_hot(by, num_classes=NUM_CLASSES).float()
            ai = torch.cat([bx]+buv, dim=0); at = torch.cat([loh]+[pseudo]*len(buv), dim=0)
            si = torch.randperm(ai.size(0))
            mx, my = mixup(ai, at, ai[si], at[si], alpha=MIXUP_ALPHA)
            n = bx.size(0); xp, yp = mx[:n], my[:n]; up, uy = mx[n:], my[n:]
            lx = -torch.mean(torch.sum(yp * F.log_softmax(mm_model(xp), dim=1), dim=1))
            lu = F.mse_loss(F.softmax(mm_model(up), dim=1), uy)
            loss = lx + LAMBDA_U * lu
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            tlx += lx.item(); tlu += lu.item(); nb += 1
        mm_model.eval(); correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                correct += (mm_model(imgs).argmax(1) == labels).sum().item()
                total += labels.size(0)
        val_acc = 100 * correct / total
        hist["epoch"].append(epoch+1); hist["loss_x"].append(tlx/nb); hist["loss_u"].append(tlu/nb); hist["val_acc"].append(val_acc)
        if val_acc > best: best = val_acc; best_state = copy.deepcopy(mm_model.state_dict())
    mm_model.load_state_dict(best_state)
    _saved_SEED = SEED; globals()["SEED"] = SEED_CUR
    full_evaluation(mm_model, val_loader, base_dataset.classes, "B4_semi_supervised",
                    hist, f"{CUR_OUT}/B4_semi_supervised", "10pct")
    globals()["SEED"] = _saved_SEED
    print(f" best={best:.2f}%  ({(time.time()-t0)/60:.1f}m)")
    del mm_model, optimizer; empty_cache()

    # ----- B5: Hybrid -----
    set_seed(SEED_CUR)
    simclr_clean = SimCLRModel(feature_dim=128, init="imagenet").to(device)
    simclr_clean.load_state_dict(simclr_ckpt["model_state_dict"])
    class SimCLRMixMatchFull(nn.Module):
        def __init__(self, enc, n): super().__init__(); self.encoder = copy.deepcopy(enc); self.classifier = nn.Linear(512, n)
        def forward(self, x): return self.classifier(self.encoder(x))
    b5 = SimCLRMixMatchFull(simclr_clean.encoder, NUM_CLASSES).to(device)
    optimizer = torch.optim.Adam(b5.parameters(), lr=1e-4)
    hist = {"epoch": [], "loss_x": [], "loss_u": [], "val_acc": []}
    best, best_state = 0.0, None
    print(f"  Training B5 (seed {SEED_CUR})...", end="", flush=True)
    t0 = time.time()
    for epoch in range(EPOCHS_DEFAULT):
        b5.train(); tlx, tlu, nb = 0.0, 0.0, 0
        uiter = iter(unlabeled_loader)
        for bx, by in labeled_loader:
            try: buv = next(uiter)
            except StopIteration: uiter = iter(unlabeled_loader); buv = next(uiter)
            bx, by = bx.to(device), by.to(device); buv = [v.to(device) for v in buv]
            with torch.no_grad():
                probs = torch.zeros(buv[0].size(0), NUM_CLASSES, device=device)
                for v in buv: probs += F.softmax(b5(v), dim=1)
                probs /= len(buv); pseudo = sharpen(probs, T=SHARPEN_TEMP)
            loh = F.one_hot(by, num_classes=NUM_CLASSES).float()
            ai = torch.cat([bx]+buv, dim=0); at = torch.cat([loh]+[pseudo]*len(buv), dim=0)
            si = torch.randperm(ai.size(0))
            mx, my = mixup(ai, at, ai[si], at[si], alpha=MIXUP_ALPHA)
            n = bx.size(0); xp, yp = mx[:n], my[:n]; up, uy = mx[n:], my[n:]
            lx = -torch.mean(torch.sum(yp * F.log_softmax(b5(xp), dim=1), dim=1))
            lu = F.mse_loss(F.softmax(b5(up), dim=1), uy)
            loss = lx + LAMBDA_U * lu
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            tlx += lx.item(); tlu += lu.item(); nb += 1
        b5.eval(); correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                correct += (b5(imgs).argmax(1) == labels).sum().item()
                total += labels.size(0)
        val_acc = 100 * correct / total
        hist["epoch"].append(epoch+1); hist["loss_x"].append(tlx/nb); hist["loss_u"].append(tlu/nb); hist["val_acc"].append(val_acc)
        if val_acc > best: best = val_acc; best_state = copy.deepcopy(b5.state_dict())
    b5.load_state_dict(best_state)
    _saved_SEED = SEED; globals()["SEED"] = SEED_CUR
    full_evaluation(b5, val_loader, base_dataset.classes, "B5_hybrid",
                    hist, f"{CUR_OUT}/B5_hybrid", "10pct")
    globals()["SEED"] = _saved_SEED
    print(f" best={best:.2f}%  ({(time.time()-t0)/60:.1f}m)")
    del b5, optimizer, simclr_clean; empty_cache()

print(f"\n{'='*80}\nAug-Naive sweep done. Total: {(time.time()-overall_start)/60:.1f} min\n{'='*80}")

Multi-seed sweep on Aug-Naive (6 classes, 4976 images)
Seeds: [123, 456]


████████████████████████████████████████████████████████████████████████████████
█  AUG-NAIVE  ·  SEED 123
████████████████████████████████████████████████████████████████████████████████
  Split: lab=495  unl=3986  val=495
  Leakage: 151 of 385 val sources have a sibling in train (39.2%)

  Training B1 (seed 123)... best=82.22%  (2.2m)


Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor


  Training B3 (seed 123)... best=85.45%  (2.3m)
  Training B4 (seed 123)... best=85.66%  (5.5m)
  Training B5 (seed 123)... best=84.24%  (5.4m)

████████████████████████████████████████████████████████████████████████████████
█  AUG-NAIVE  ·  SEED 456
████████████████████████████████████████████████████████████████████████████████
  Split: lab=495  unl=3986  val=495
  Leakage: 166 of 399 val sources have a sibling in train (41.6%)

  Training B1 (seed 456)... best=79.39%  (2.2m)
  Training B3 (seed 456)... best=80.20%  (2.2m)
  Training B4 (seed 456)... best=81.21%  (5.4m)
  Training B5 (seed 456)... best=80.00%  (5.3m)

Aug-Naive sweep done. Total: 30.5 min


In [85]:
# ==============================================================================
# MULTI-SEED SWEEP — Aug-Grouped (Augmented Taiwan + grouped split, leak-free)
# Seeds 123 and 456.
# ==============================================================================
SEEDS_TO_RUN = [123, 456]

print(f"Multi-seed sweep on Aug-Grouped")
print(f"Seeds: {SEEDS_TO_RUN}\n")

overall_start = time.time()

for SEED_CUR in SEEDS_TO_RUN:
    print("\n" + "█" * 80)
    print(f"█  AUG-GROUPED  ·  SEED {SEED_CUR}")
    print("█" * 80)

    CUR_OUT = f"{PROJECT_DIR}/RESULTS_TAIWAN_AUG_GROUPED/10pct_seed{SEED_CUR}"
    for method in ["B1_supervised", "B3_self_supervised",
                   "B4_semi_supervised", "B5_hybrid"]:
        os.makedirs(f"{CUR_OUT}/{method}", exist_ok=True)

    # ----- Grouped split with this seed -----
    set_seed(SEED_CUR)
    labeled_idx, unlabeled_idx, val_idx = grouped_stratified_split(
        base_dataset, label_pct=0.10, val_pct=0.10, min_per_class=10, seed=SEED_CUR,
    )

    def src_of(idx):
        path, _ = base_dataset.samples[idx]
        return source_id_from_filename(os.path.basename(path))
    overlap = set(src_of(i) for i in labeled_idx) & set(src_of(i) for i in val_idx)
    print(f"  Split: lab={len(labeled_idx)}  unl={len(unlabeled_idx)}  val={len(val_idx)}")
    print(f"  Leakage check: overlap={len(overlap)} (must be 0)")
    assert len(overlap) == 0, "Grouped split has overlap — bug!"

    labeled_dataset   = LabeledSubset(base_dataset, labeled_idx,   weak_transform)
    unlabeled_dataset = UnlabeledKAug(base_dataset, unlabeled_idx, weak_transform, K=K_AUGMENTS)
    val_dataset       = LabeledSubset(base_dataset, val_idx,       eval_transform)
    labeled_loader = DataLoader(labeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
    unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    train_loader = DataLoader(labeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)

    # ----- B1 -----
    set_seed(SEED_CUR)
    sup_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    sup_model.fc = nn.Linear(512, NUM_CLASSES); sup_model = sup_model.to(device)
    optimizer = torch.optim.Adam(sup_model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    hist = {"epoch": [], "train_loss": [], "val_acc": []}
    best, best_state = 0.0, None
    print(f"  Training B1 (seed {SEED_CUR})...", end="", flush=True)
    t0 = time.time()
    for epoch in range(EPOCHS_DEFAULT):
        sup_model.train(); tl, nb = 0.0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            loss = criterion(sup_model(imgs), labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            tl += loss.item(); nb += 1
        sup_model.eval(); correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                correct += (sup_model(imgs).argmax(1) == labels).sum().item()
                total += labels.size(0)
        val_acc = 100 * correct / total
        hist["epoch"].append(epoch+1); hist["train_loss"].append(tl/nb); hist["val_acc"].append(val_acc)
        if val_acc > best: best = val_acc; best_state = copy.deepcopy(sup_model.state_dict())
    sup_model.load_state_dict(best_state)
    _saved_SEED = SEED; globals()["SEED"] = SEED_CUR
    full_evaluation(sup_model, val_loader, base_dataset.classes, "B1_supervised",
                    hist, f"{CUR_OUT}/B1_supervised", "10pct")
    globals()["SEED"] = _saved_SEED
    print(f" best={best:.2f}%  ({(time.time()-t0)/60:.1f}m)")
    del sup_model, optimizer; empty_cache()

    # ----- B3 -----
    set_seed(SEED_CUR)
    simclr_clean = SimCLRModel(feature_dim=128, init="imagenet").to(device)
    simclr_clean.load_state_dict(simclr_ckpt["model_state_dict"])
    class SimCLRFineTune(nn.Module):
        def __init__(self, enc, n): super().__init__(); self.encoder = copy.deepcopy(enc); self.classifier = nn.Linear(512, n)
        def forward(self, x): return self.classifier(self.encoder(x))
    ft_model = SimCLRFineTune(simclr_clean.encoder, NUM_CLASSES).to(device)
    optimizer = torch.optim.Adam(ft_model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    hist = {"epoch": [], "train_loss": [], "val_acc": []}
    best, best_state = 0.0, None
    print(f"  Training B3 (seed {SEED_CUR})...", end="", flush=True)
    t0 = time.time()
    for epoch in range(EPOCHS_DEFAULT):
        ft_model.train(); tl, nb = 0.0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            loss = criterion(ft_model(imgs), labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            tl += loss.item(); nb += 1
        ft_model.eval(); correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                correct += (ft_model(imgs).argmax(1) == labels).sum().item()
                total += labels.size(0)
        val_acc = 100 * correct / total
        hist["epoch"].append(epoch+1); hist["train_loss"].append(tl/nb); hist["val_acc"].append(val_acc)
        if val_acc > best: best = val_acc; best_state = copy.deepcopy(ft_model.state_dict())
    ft_model.load_state_dict(best_state)
    _saved_SEED = SEED; globals()["SEED"] = SEED_CUR
    full_evaluation(ft_model, val_loader, base_dataset.classes, "B3_self_supervised",
                    hist, f"{CUR_OUT}/B3_self_supervised", "10pct")
    globals()["SEED"] = _saved_SEED
    print(f" best={best:.2f}%  ({(time.time()-t0)/60:.1f}m)")
    del ft_model, optimizer, simclr_clean; empty_cache()

    # ----- B4 -----
    set_seed(SEED_CUR)
    mm_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    mm_model.fc = nn.Linear(512, NUM_CLASSES); mm_model = mm_model.to(device)
    optimizer = torch.optim.Adam(mm_model.parameters(), lr=1e-4)
    hist = {"epoch": [], "loss_x": [], "loss_u": [], "val_acc": []}
    best, best_state = 0.0, None
    print(f"  Training B4 (seed {SEED_CUR})...", end="", flush=True)
    t0 = time.time()
    for epoch in range(EPOCHS_DEFAULT):
        mm_model.train(); tlx, tlu, nb = 0.0, 0.0, 0
        uiter = iter(unlabeled_loader)
        for bx, by in labeled_loader:
            try: buv = next(uiter)
            except StopIteration: uiter = iter(unlabeled_loader); buv = next(uiter)
            bx, by = bx.to(device), by.to(device); buv = [v.to(device) for v in buv]
            with torch.no_grad():
                probs = torch.zeros(buv[0].size(0), NUM_CLASSES, device=device)
                for v in buv: probs += F.softmax(mm_model(v), dim=1)
                probs /= len(buv); pseudo = sharpen(probs, T=SHARPEN_TEMP)
            loh = F.one_hot(by, num_classes=NUM_CLASSES).float()
            ai = torch.cat([bx]+buv, dim=0); at = torch.cat([loh]+[pseudo]*len(buv), dim=0)
            si = torch.randperm(ai.size(0))
            mx, my = mixup(ai, at, ai[si], at[si], alpha=MIXUP_ALPHA)
            n = bx.size(0); xp, yp = mx[:n], my[:n]; up, uy = mx[n:], my[n:]
            lx = -torch.mean(torch.sum(yp * F.log_softmax(mm_model(xp), dim=1), dim=1))
            lu = F.mse_loss(F.softmax(mm_model(up), dim=1), uy)
            loss = lx + LAMBDA_U * lu
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            tlx += lx.item(); tlu += lu.item(); nb += 1
        mm_model.eval(); correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                correct += (mm_model(imgs).argmax(1) == labels).sum().item()
                total += labels.size(0)
        val_acc = 100 * correct / total
        hist["epoch"].append(epoch+1); hist["loss_x"].append(tlx/nb); hist["loss_u"].append(tlu/nb); hist["val_acc"].append(val_acc)
        if val_acc > best: best = val_acc; best_state = copy.deepcopy(mm_model.state_dict())
    mm_model.load_state_dict(best_state)
    _saved_SEED = SEED; globals()["SEED"] = SEED_CUR
    full_evaluation(mm_model, val_loader, base_dataset.classes, "B4_semi_supervised",
                    hist, f"{CUR_OUT}/B4_semi_supervised", "10pct")
    globals()["SEED"] = _saved_SEED
    print(f" best={best:.2f}%  ({(time.time()-t0)/60:.1f}m)")
    del mm_model, optimizer; empty_cache()

    # ----- B5 -----
    set_seed(SEED_CUR)
    simclr_clean = SimCLRModel(feature_dim=128, init="imagenet").to(device)
    simclr_clean.load_state_dict(simclr_ckpt["model_state_dict"])
    class SimCLRMixMatchFull(nn.Module):
        def __init__(self, enc, n): super().__init__(); self.encoder = copy.deepcopy(enc); self.classifier = nn.Linear(512, n)
        def forward(self, x): return self.classifier(self.encoder(x))
    b5 = SimCLRMixMatchFull(simclr_clean.encoder, NUM_CLASSES).to(device)
    optimizer = torch.optim.Adam(b5.parameters(), lr=1e-4)
    hist = {"epoch": [], "loss_x": [], "loss_u": [], "val_acc": []}
    best, best_state = 0.0, None
    print(f"  Training B5 (seed {SEED_CUR})...", end="", flush=True)
    t0 = time.time()
    for epoch in range(EPOCHS_DEFAULT):
        b5.train(); tlx, tlu, nb = 0.0, 0.0, 0
        uiter = iter(unlabeled_loader)
        for bx, by in labeled_loader:
            try: buv = next(uiter)
            except StopIteration: uiter = iter(unlabeled_loader); buv = next(uiter)
            bx, by = bx.to(device), by.to(device); buv = [v.to(device) for v in buv]
            with torch.no_grad():
                probs = torch.zeros(buv[0].size(0), NUM_CLASSES, device=device)
                for v in buv: probs += F.softmax(b5(v), dim=1)
                probs /= len(buv); pseudo = sharpen(probs, T=SHARPEN_TEMP)
            loh = F.one_hot(by, num_classes=NUM_CLASSES).float()
            ai = torch.cat([bx]+buv, dim=0); at = torch.cat([loh]+[pseudo]*len(buv), dim=0)
            si = torch.randperm(ai.size(0))
            mx, my = mixup(ai, at, ai[si], at[si], alpha=MIXUP_ALPHA)
            n = bx.size(0); xp, yp = mx[:n], my[:n]; up, uy = mx[n:], my[n:]
            lx = -torch.mean(torch.sum(yp * F.log_softmax(b5(xp), dim=1), dim=1))
            lu = F.mse_loss(F.softmax(b5(up), dim=1), uy)
            loss = lx + LAMBDA_U * lu
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            tlx += lx.item(); tlu += lu.item(); nb += 1
        b5.eval(); correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                correct += (b5(imgs).argmax(1) == labels).sum().item()
                total += labels.size(0)
        val_acc = 100 * correct / total
        hist["epoch"].append(epoch+1); hist["loss_x"].append(tlx/nb); hist["loss_u"].append(tlu/nb); hist["val_acc"].append(val_acc)
        if val_acc > best: best = val_acc; best_state = copy.deepcopy(b5.state_dict())
    b5.load_state_dict(best_state)
    _saved_SEED = SEED; globals()["SEED"] = SEED_CUR
    full_evaluation(b5, val_loader, base_dataset.classes, "B5_hybrid",
                    hist, f"{CUR_OUT}/B5_hybrid", "10pct")
    globals()["SEED"] = _saved_SEED
    print(f" best={best:.2f}%  ({(time.time()-t0)/60:.1f}m)")
    del b5, optimizer, simclr_clean; empty_cache()

print(f"\n{'='*80}\nAug-Grouped sweep done. Total: {(time.time()-overall_start)/60:.1f} min\n{'='*80}")

Multi-seed sweep on Aug-Grouped
Seeds: [123, 456]


████████████████████████████████████████████████████████████████████████████████
█  AUG-GROUPED  ·  SEED 123
████████████████████████████████████████████████████████████████████████████████
  Split: lab=507  unl=3965  val=504
  Leakage check: overlap=0 (must be 0)
  Training B1 (seed 123)... best=65.67%  (2.2m)
  Training B3 (seed 123)... best=64.88%  (2.2m)
  Training B4 (seed 123)... best=70.44%  (5.5m)
  Training B5 (seed 123)... best=64.88%  (5.5m)

████████████████████████████████████████████████████████████████████████████████
█  AUG-GROUPED  ·  SEED 456
████████████████████████████████████████████████████████████████████████████████
  Split: lab=510  unl=3966  val=500
  Leakage check: overlap=0 (must be 0)
  Training B1 (seed 456)... best=69.40%  (2.3m)
  Training B3 (seed 456)... best=67.80%  (2.2m)
  Training B4 (seed 456)... best=72.40%  (5.5m)
  Training B5 (seed 456)... best=68.40%  (5.5m)

Aug-Grouped sweep done. Total: 3